# Когда алгоритм и человек расходятся — кто прав?

Стандартный детектор точек разладки (PELT), прогнанный по истории прослушиваний за несколько лет, вернул 7 дат. Один период, который уверенно вспоминается как переломный, в список не попал.

Есть два одинаково бесполезных способа отреагировать на такое расхождение: поверить алгоритму (память субъективна) или не поверить алгоритму (метод мог упустить). Оба варианта — это выбор без проверки. Вместо выбора — построена независимая проверка, которая отвечает на вопрос «кто прав» фактами, а не доверием к одной из сторон.

Результат: алгоритм не ошибся в расчётах — он искал сдвиг не в том признаковом пространстве (настроение музыки вместо состава артистов вокруг человека в этот период). Отдельный метод, спроектированный под эту гипотезу, нашёл сигнал.

Это стало не разовым фиксом, а причиной пересобрать весь пайплайн заново — вместо одного детектора получилось четыре независимых слоя точек и отдельная ось Variability, каждый проверяет свою природу сигнала:

| Детектор | База | Найдено |
|---|---|---|
| PELT-A (звук) | all | 3 точки |
| PELT-B (поведение) | all | 5 точек |
| Cosine (состав артистов) | all | 3 точки |
| Storm (зоны нестабильности) | all | 2 эпизода |
| Variability (ось компаса) | valid | 0.417, Medium |

11 отметок по слоям (9 уникальных дат, 2024-02 видна трём слоям) и 2 шторма вместо 7 непроверенных дат одного метода — и, что важнее числа, у каждой известно, **почему** она найдена и насколько ей можно доверять. Дальше — как строился этот путь, шаг за шагом.



## Методология поверх личных данных Spotify

Прикладной кейс о построении статистически защищённых выводов на шумных, разнородных временных рядах при N=1 — когда единственный способ проверить метод — независимая content-проверка, а не более сложная математика.

### Стек и методы

- **Change-point detection:** `ruptures` (PELT, L2), BIC-калибровка штрафа, sensitivity-анализ по сетке `pen`.
- **Устойчивость:** bootstrap с сохранением временного порядка (субсэмплинг 85%, 500 итераций) — отдельно проверяет устойчивость к гиперпараметру и к вариации данных.
- **Multivariate distance:** Mahalanobis по ковариации разностей (не уровней) с Ledoit-Wolf shrinkage.
- **Identity-shift detection:** косинусное расстояние на категориальных share-векторах — там, где permutation-null оказался неприменим (продемонстрировано на данных, объяснено структурой признака), возвращена MAD-детекция пиков.
- **Permutation testing:** нулевые распределения ресэмплингом реальных наблюдений; двойной гейт «статистическая + практическая значимость» — обязателен при больших N, иначе значимым становится любое малое отклонение.
- **Detrending:** каждая межосевая корреляция проверяется на артефакт общего тренда перед интерпретацией.

### Инженерный результат

Метод обобщён в переиспользуемый модуль (`VariabilityAnalyzer`) с явным API и задокументированным принципом «единицы обмена» — применим к любым данным вида «период → наблюдения → агрегат», не только к музыке.

### Что доказано, а не предположено

Self-percentile математически не различает людей между собой — показано синтетическим тестом до применения к реальным данным. Сильнейшая корреляция всего анализа (`r=-0.55, p<0.0001`) не пережила проверку на тренд-артефакт. Периодизация истории на «стабильные эпохи» не подтвердилась и зафиксирована как отрицательный результат, а не скрыта. Bootstrap с сохранением временного порядка отделил устойчивые точки от артефактных: из 7 наивных выжило 5 — **на смешанном наборе из 7 признаков**. Ещё одна точка позже была найдена отдельным методом (Cosine) — сам по себе bootstrap на одном слое не финальная инстанция.

### Что не вошло

Методы, проверенные и отброшенные: **self-percentile** и **периодизация** (фазы 5–6), **SNR-фильтрация** (приложение A), **cut-scan как детектор точек** (приложение B), **permutation-null для категориальных признаков** (приложение C). Каждый — с причиной отбраковки.

---


In [1]:
# 0.1 ИМПОРТЫ И ПОДКЛЮЧЕНИЕ src/
# ============================================================================
# Единственное место, где импортируются модули. Все остальные ячейки
# используют то, что загружено здесь. Абсолютных путей в ноутбуке нет.
# ============================================================================

import sys
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings('ignore', category=RuntimeWarning,
                        message='All-NaN slice encountered')


def find_src(start=None):
    start = Path(start or Path.cwd()).resolve()
    for p in [start] + list(start.parents):
        # Маркер: src/ с ключевыми модулями проекта
        if (p / 'src' / 'paths.py').is_file() and \
           (p / 'src' / 'loaders.py').is_file():
            return p / 'src'
    raise FileNotFoundError(
        'src/ не найден выше текущей директории. '
        'Ожидается структура: <project_root>/src/{paths.py,loaders.py,...}'
    )

SRC_PATH = find_src()
if str(SRC_PATH) not in sys.path:
    sys.path.append(str(SRC_PATH))
if str(SRC_PATH / 'detectors') not in sys.path:
    sys.path.append(str(SRC_PATH / 'detectors'))

from paths import get_base_path, get_dashboard_path
from loaders import load_all
from temporal import compute_windows, trim_cold_start
from aggregators.music import build_monthly_sound, SOUND_FEATURES
from aggregators.behavior import build_monthly_behavior

print(f"✅ src/ подключён:")

✅ src/ подключён:


In [2]:
# 0.2 ЗАГРУЗКА ДАННЫХ
# ============================================================================

BASE_PATH = get_base_path()
DASHBOARD_PATH = get_dashboard_path()

data = load_all(BASE_PATH)
df = data['df']
playlist_links = data['playlist_links']
library_tracks = data['library_tracks']
owner_ids = data['owner_ids']

# Обогащение — то же, что в преп-ноуте
from enrich import (
    add_timestamps, add_valid_play, add_first_valid_seen,
    add_my_music, add_sessions, add_track_source,
)
df = add_timestamps(df)
df = add_valid_play(df)
df = add_first_valid_seen(df)
df, my_music_uris = add_my_music(df, library_tracks, playlist_links, owner_ids)
df = add_sessions(df)
df = add_track_source(df, library_tracks, playlist_links, owner_ids)

meta = compute_windows(df, playlist_links)
windows = meta['windows']

print(f"✅ Данные: {len(df):,} событий, "
      f"{df['period'].nunique()} месяцев")
print(f"   Период: {df['period'].min()} — {df['period'].max()}")

✅ Данные: 79,752 событий, 72 месяцев
   Период: 2020-05 — 2026-08


## Карта расследования

| # | Фаза | Что проверялось | Исход |
|---|---|---|---|
| 1 | Наивный старт | `pen=10` + 7 смешанных признаков | ❌ сломалось |
| 2 | Bootstrap | Устойчивость точек к пересэмплированию | 🔄 зоны, не точки |
| 3 | Cosine | Смена артистов при похожем звуке | ✅ подтвердилось |
| 4 | Разделение слоёв | Звук и поведение — один процесс | ✅ два разных |
| 5 | Периодизация | Стабильные эпохи между точками | ❌ 0 из 6 |
| 6 | Self-percentile | Свой ранг как уровень | ❌ структурно невозможен |
| 7 | Noise-floor | Permutation-нуль для Variability | ✅ работает |
| 8 | Разные нули | Один null на все признаки | 🔄 свой под структуру |
| 9 | Detrend | Корреляции осей реальны | 🔄 2 не установлены |
| 10 | Behavior в компас | Связь с Burnout | ⚠️ не входит в компас |
| 11 | Toolkit | Обобщение в модуль | ✅ |
| 12 | ALL vs VALID | Одна база для всего | 🔄 по семантике |

**Легенда:** ✅ подтвердилось · 🔄 поворот · ❌ провал · 
⚠️ архитектурное ограничение

**Приложения:** SNR, Cut-scan, Storm — фоновые проверки, 
не вошедшие в основной нарратив.

## Как устроен этот ноутбук

**Структура:**

- **Вводная** — с чего началось, что стало результатом.
- **12 фаз** — каждая = гипотеза → проверка → результат → поворот → 
  что дальше.
- **Финал** — что в продакшн, что отброшено, что под вопросом.
- **Приложения A/B/C** — фоновые проверки.

**Формат каждой фазы:**

> **Гипотеза** — что ожидалось.  
> **Что делали** — как проверялось.  
> **Результат** — числа.  
> 🔄 **Поворот** — что оказалось иначе.  
> **→ Ведёт к** — как повлияло на следующие фазы.

**Ключевые места** Каждая фаза заканчивается блоком **🔄 Поворот** — вместе они каркас истории., они — каркас 
истории.



# Фаза 1. Наивный старт


## Фаза 1. Наивный старт

### Гипотеза

Точки разладки можно найти одним PELT на всех доступных признаках: 
4 звуковых (`valence`, `energy`, `danceability`, `instrumentalness`) 
и 3 поведенческих (`skip_rate`, `played_frac`, `orphan_share`).

### Метод

`ruptures.Pelt(model="l2")`, штраф `pen=10`. Классификация сценария 
по числу точек в год: ≤0.5 — single, 0.5–1.0 — moderate, >1.0 — cascade.




In [3]:
# 1 ФАЗА 1: НАИВНЫЙ СТАРТ
# ============================================================================
# Воспроизведение подхода из 07 ноутбука (cell 4.5).
# Смешанные признаки, pen=10 на глаз, пороги на глаз.
# ============================================================================
import ruptures as rpt

PELT_PENALTY = 10
LOW_VARIABILITY_THRESHOLD = 0.5
MODERATE_VARIABILITY_THRESHOLD = 1.0
METRICS_NAIVE = ['valence', 'energy', 'danceability', 'instrumentalness',
                 'skip_rate', 'played_frac', 'orphan_share']

# Месячные ряды
df_naive = df.copy()
df_naive['ts_naive'] = pd.to_datetime(df_naive['ts_naive'], errors='coerce')

monthly_naive = df_naive.groupby('period').agg(
    valence=('valence', 'mean'),
    energy=('energy', 'mean'),
    danceability=('danceability', 'mean'),
    instrumentalness=('instrumentalness', 'mean'),
    skip_rate=('is_skip', 'mean'),
    played_frac=('played_frac', 'mean'),
    orphan_share=('my_music', lambda x: 1 - x.mean()),
).reset_index().rename(columns={'period': 'month'})

# Исключаем неполный последний месяц
max_ts = df_naive['ts_naive'].max()
last_month = max_ts.to_period('M')
if max_ts.day < last_month.days_in_month:
    monthly_naive = monthly_naive[monthly_naive['month'] < last_month]

monthly_naive = monthly_naive.set_index('month').sort_index()

# Заполняем пропуски
for col in METRICS_NAIVE:
    if col in ['skip_rate', 'orphan_share']:
        monthly_naive[col] = monthly_naive[col].fillna(0)
    else:
        monthly_naive[col] = monthly_naive[col].fillna(
            monthly_naive[col].median()
        )

# PELT
X = monthly_naive[METRICS_NAIVE].values
mean, std = X.mean(axis=0), X.std(axis=0)
std[std == 0] = 1
X_norm = (X - mean) / std

model = rpt.Pelt(model="l2").fit(X_norm)
cp_indices = model.predict(pen=PELT_PENALTY)[:-1]
cp_dates = [monthly_naive.index[i].strftime('%Y-%m') for i in cp_indices]

n_points = len(cp_indices)
total_months = len(monthly_naive)
points_per_year = n_points / (total_months / 12)

print(f"🔍 Точек разладки: {n_points}")
print(f"   Даты: {cp_dates}")
print(f"   Точек в год: {points_per_year:.2f}")

if points_per_year <= LOW_VARIABILITY_THRESHOLD:
    scenario, variability = 'single', 'Low'
elif points_per_year <= MODERATE_VARIABILITY_THRESHOLD:
    scenario, variability = 'moderate', 'Medium'
else:
    scenario, variability = 'cascade', 'High'

print(f"   Сценарий: {scenario}")
print(f"   Variability: {variability}")

🔍 Точек разладки: 7
   Даты: ['2021-07', '2022-05', '2022-10', '2023-08', '2024-06', '2024-11', '2025-09']
   Точек в год: 1.18
   Сценарий: cascade
   Variability: High


### Результат

7 точек, сценарий cascade, Variability = High.
2021-07, 2022-05, 2022-10, 2023-08, 2024-06, 2024-11, 2025-09


### 🔄 Поворот

Гипотеза «одна модель на всё» не выдержала. Три независимых причины:

1. **Смешаны звук и поведение.** Разные процессы глушат друг друга: 
   один шумный признак поднимает штраф для всех, включая чистые.
2. **`pen=10` не выводится из данных.** Sensitivity-анализ при 
   `pen=5..30` даёт разные сценарии: cascade / moderate / single. 
   Точки не воспроизводятся при разумных колебаниях штрафа.
3. **`orphan_share` семантически связан с Discovery.** Дублирование 
   признака в модели.

### → Ведёт к

Фаза 2: нужен принципиальный штраф, а не подобранный. Плюс проверка 
устойчивости — точки PELT сами по себе ничего не доказывают.


# Фаза 2. Bootstrap устойчивости


## Фаза 2. Bootstrap устойчивости

### Гипотеза

7 точек из фазы 1 — реальные разладки, а не артефакт `pen=10` 
или случайных выбросов в конкретные месяцы.

### Метод

Bootstrap с сохранением временного порядка: 500 итераций, в каждой 
случайно исключается 15% месяцев, PELT пересчитывается с тем же 
`pen=10`. Фильтр устойчивости: `stability ≥ 0.10`.

Две разные проверки:
- **Переживает ли точка вариацию `pen`** — sensitivity-анализ.
- **Переживает ли она вариацию данных** — bootstrap.



In [4]:
# 2. Фаза 2. Bootstrap устойчивости
# ============================================================================
# Точка PELT без bootstrap — не точка. Пересэмплируем 15% месяцев,
# смотрим, какие точки воспроизводятся. Фильтр: stability >= 0.10.
# Плюс bar chart — видно, что топ — пары соседних месяцев, не точки.
# ============================================================================
import ruptures as rpt
import plotly.graph_objects as go

N_BOOT = 500
DROP_FRAC = 0.15
STABILITY_THRESHOLD = 0.10

# --- Bootstrap ---
rng = np.random.default_rng(42)
n_months = len(monthly_naive)
n_keep = int(round(n_months * (1 - DROP_FRAC)))

counts = {}
for _ in range(N_BOOT):
    keep_idx = np.sort(rng.choice(n_months, size=n_keep, replace=False))
    X_sub = X_norm[keep_idx]
    bkps = rpt.Pelt(model="l2").fit(X_sub).predict(pen=PELT_PENALTY)
    for i in bkps[:-1]:
        d = monthly_naive.index[keep_idx[i]].strftime('%Y-%m')
        counts[d] = counts.get(d, 0) + 1

stability = pd.Series(counts).sort_values(ascending=False) / N_BOOT

# --- Статус точек из фазы 1 ---
print(f"📊 Bootstrap ({N_BOOT} итераций, drop {DROP_FRAC:.0%}):\n")
for d in cp_dates:
    s = stability.get(d, 0.0)
    mark = '✅' if s >= STABILITY_THRESHOLD else '⚠️'
    print(f"   {d}: {s:.3f} {mark}")

points_filtered = [d for d in cp_dates
                   if stability.get(d, 0) >= STABILITY_THRESHOLD]
print(f"\n🎯 После фильтра: {len(points_filtered)} из {len(cp_dates)}")
print(f"   {points_filtered}")

# --- Визуализация: топ-15 ---
# --- Визуализация: топ-15, с выделением исходных 7 ---
top15 = stability.head(15).sort_values(ascending=True)
labels_y = [str(d) for d in top15.index]
colors = ['#d62728' if str(d) in cp_dates else '#1f77b4'
          for d in top15.index]

fig = go.Figure()
fig.add_trace(go.Bar(
    x=top15.values,
    y=labels_y,
    orientation='h',
    text=[f'{v:.3f}' for v in top15.values],
    textposition='outside',
    marker_color=colors,
    marker_line_width=0,
))

fig.add_vline(
    x=STABILITY_THRESHOLD,
    line_dash='dash', line_color='gray',
    annotation_text=f'порог {STABILITY_THRESHOLD}',
    annotation_position='top right',
)

fig.update_layout(
    template='plotly_white',
    height=550,
    title=f'Bootstrap устойчивость (n={N_BOOT}, drop {DROP_FRAC:.0%})<br>'
          f'<sub>Красные — из 7 исходных точек PELT. Синие — bootstrap нашёл сам</sub>',
    xaxis_title='Stability',
    showlegend=False,
    margin=dict(l=80, r=40, t=80, b=40),
)
fig.update_xaxes(range=[0, top15.max() * 1.15])
fig.update_yaxes(type='category', autorange='reversed')
fig.show()

📊 Bootstrap (500 итераций, drop 15%):

   2021-07: 0.124 ✅
   2022-05: 0.078 ⚠️
   2022-10: 0.158 ✅
   2023-08: 0.212 ✅
   2024-06: 0.100 ✅
   2024-11: 0.022 ⚠️
   2025-09: 0.270 ✅

🎯 После фильтра: 5 из 7
   ['2021-07', '2022-10', '2023-08', '2024-06', '2025-09']


### Результат

Из 7 исходных точек выжили **5**. Две отсеялись:

- `2022-05` — 0.078
- `2024-11` — 0.022 (артефакт)

Топ-bootstrap — **пары соседних месяцев** (2025-08/2025-09, 
2023-08/2023-09). Сигнал размазан, не точки.

Bootstrap выявил **~10 устойчивых месяцев**, которых `pen=10` не 
увидел (`2021-08` = 0.250, `2024-03` = 0.204).

### 🔄 Поворот

**Точка PELT — не точка.** Три следствия:

1. **`pen=10` пропускает реальные разладки.** Bootstrap нашёл 
   минимум 10 устойчивых месяцев, которых не было в исходных 7.
2. **`pen=10` добавляет случайные.** `2024-11` (0.022) и `2022-05` 
   (0.078) — артефакты конкретного набора данных.
3. **Сигнал размазан.** Топ-bootstrap — пары соседних месяцев. 
   Реальная разладка — не «сентябрь 2025», а «зона август–сентябрь».

**Оговорка про `2022-05`.** Bootstrap на **смешанном наборе из 7 признаков** её отсеял. В фазе 3 точка будет найдена отдельным методом (Cosine) 
с устойчивостью 0.825. Bootstrap на одном слое — не финальная 
инстанция.

### → Ведёт к

Фаза 3: PELT-A на звуке **не видит** смену артистов при похожем 
звучании. Нужен отдельный слой. Плюс: раз точка «плавает» вокруг 
реального перелома, нужен принципиальный штраф (BIC), а не 
подобранный.


# Фаза 3. Cosine — смена идентичности


## Фаза 3. Cosine — смена идентичности

### Гипотеза

Наивный PELT из фазы 1 смотрит на среднее по признакам. Если 
Radiohead сменился на Portishead при похожих `valence`/`energy` — 
метод этого не увидит. Средние не изменились, а кто играет — 
изменился.

Нужен отдельный слой: косинусное расстояние по составу артистов 
между соседними месяцами.

### Метод

Матрица «месяц × артист» с долями. Косинусное расстояние между 
соседними месяцами: 0 — идентичный состав, 1 — полностью разные. 
Пики через `медиана + 2·MAD`. Bootstrap устойчивости.

Для контраста — та же логика на **жанрах**. Ожидание: жанры инертны, 
дадут шум.

In [5]:
# 3. Фаза 3. Cosine — смена идентичности
# ============================================================================
# Косинусное расстояние по составу артистов между соседними месяцами.
# Пики через медиана + 2·MAD. Bootstrap устойчивости.
# Плюс — проверка на жанрах (контраст: они не работают).
# ============================================================================
import plotly.graph_objects as go


def monthly_share_matrix(df_input, id_col, explode=False, min_events=10):
    d = df_input[['period', id_col]].dropna().copy()
    if explode:
        d[id_col] = d[id_col].astype(str).str.split(',')
        d = d.explode(id_col)
        d[id_col] = d[id_col].str.strip()
    d = d[d[id_col] != 'unknown']
    month_sizes = d.groupby('period').size()
    valid_months = month_sizes[month_sizes >= min_events].index
    d = d[d['period'].isin(valid_months)]
    counts = d.groupby(['period', id_col]).size().unstack(fill_value=0)
    return counts.div(counts.sum(axis=1), axis=0).sort_index()


def cosine_distance_series(shares):
    vals = shares.values
    idx = shares.index
    out = []
    for i in range(1, len(vals)):
        a, b = vals[i - 1], vals[i]
        denom = np.linalg.norm(a) * np.linalg.norm(b)
        out.append(1 - np.dot(a, b) / denom if denom > 0 else np.nan)
    return pd.Series(out, index=[idx[i].strftime('%Y-%m')
                                 for i in range(1, len(idx))])


def find_peaks(dist, k_mad=2.0):
    med = dist.median()
    mad = np.median(np.abs(dist - med))
    thr = med + k_mad * mad
    return dist[dist > thr].sort_values(ascending=False), thr


def group_peaks(peaks_dict, gap_months=3):
    if not peaks_dict:
        return {}
    periods = sorted(pd.Period(p, 'M') for p in peaks_dict.keys())
    clusters, current = [], [periods[0]]
    for p in periods[1:]:
        if (p - current[-1]).n <= gap_months:
            current.append(p)
        else:
            clusters.append(current)
            current = [p]
    clusters.append(current)
    out = {}
    for cl in clusters:
        best = max(cl, key=lambda p: peaks_dict.get(str(p), 0))
        out[str(best)] = {
            'region': f"{cl[0]}..{cl[-1]}" if len(cl) > 1 else str(cl[0]),
            'value': peaks_dict[str(best)],
            'n_peaks': len(cl),
        }
    return out


# --- Артисты ---
artist_dist = cosine_distance_series(
    monthly_share_matrix(df, 'artist_name', explode=False)
)

# --- Жанры (контраст) ---
genre_dist = cosine_distance_series(
    monthly_share_matrix(df, 'genres_normalized', explode=True)
)

# --- Пороги и пики ---
artist_peaks, thr_a = find_peaks(artist_dist)
genre_peaks, thr_g = find_peaks(genre_dist)

print(f"Порог артистов: {thr_a:.3f}, пиков: {len(artist_peaks)}")
print(f"Порог жанров:   {thr_g:.3f}, пиков: {len(genre_peaks)}")

print(f"\nПики АРТИСТОВ:")
for d, v in artist_peaks.items():
    print(f"   {d}: {v:.3f}")

print(f"\nПики ЖАНРОВ (первые 15):")
for d, v in list(genre_peaks.items())[:15]:
    print(f"   {d}: {v:.3f}")

# --- Группировка пиков артистов ≤3 мес ---
artist_grouped = group_peaks(artist_peaks.to_dict())
print(f"\nСгруппированные пики артистов ({len(artist_grouped)}):")
for k, v in artist_grouped.items():
    print(f"   {k} (регион {v['region']}, "
          f"max={v['value']:.3f}, пиков={v['n_peaks']})")

# --- Bootstrap устойчивости ---
rng = np.random.default_rng(42)
counts = {}
n_months = len(artist_dist)
n_keep = int(round(n_months * 0.85))
for _ in range(200):
    keep_idx = np.sort(rng.choice(n_months, size=n_keep, replace=False))
    sub = artist_dist.iloc[keep_idx]
    med = sub.median()
    mad = np.median(np.abs(sub - med))
    thr = med + 2 * mad
    for d in sub[sub > thr].index:
        counts[d] = counts.get(d, 0) + 1

stability_cos = pd.Series(counts).sort_values(ascending=False) / 200
print(f"\nBootstrap устойчивости (топ-10):")
print(stability_cos.head(10).round(3))

# --- Визуализация: distance во времени ---
x = pd.PeriodIndex(artist_dist.index, freq='M').to_timestamp()

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=x, y=artist_dist.values,
    mode='lines+markers',
    name='Cosine distance',
    line=dict(width=2),
    marker=dict(size=5),
))
fig.add_hline(
    y=thr_a, line_dash='dash', line_color='gray',
    annotation_text=f'порог {thr_a:.3f}',
    annotation_position='top right',
)

# Вертикальные линии на пиках
for peak in artist_peaks.index:
    fig.add_vline(
        x=pd.Period(peak, 'M').to_timestamp(),
        line_dash='dot', line_color='#d62728', opacity=0.4,
    )

fig.update_layout(
    template='plotly_white',
    height=450,
    title='Косинусное расстояние по артистам',
    xaxis_title='Месяц',
    yaxis_title='Distance',
    showlegend=False,
)
fig.show()

from detectors.cosine_identity import detect as cosine_detect

df_t_local = trim_cold_start(df)
_res_cos = cosine_detect(df_t_local, id_col='artist_name')
print("\nПродакшн-детектор (trimmed), stability:")
print(pd.Series(_res_cos['stability']).sort_values(ascending=False).head(6).round(3))

Порог артистов: 0.908, пиков: 6
Порог жанров:   0.404, пиков: 15

Пики АРТИСТОВ:
   2021-02: 0.993
   2024-02: 0.961
   2021-03: 0.958
   2022-05: 0.949
   2022-06: 0.933
   2022-09: 0.911

Пики ЖАНРОВ (первые 15):
   2021-03: 0.783
   2021-06: 0.783
   2023-12: 0.716
   2023-11: 0.712
   2021-02: 0.685
   2024-07: 0.679
   2024-08: 0.650
   2024-06: 0.615
   2021-05: 0.555
   2022-09: 0.543
   2020-12: 0.490
   2021-07: 0.467
   2024-01: 0.444
   2021-09: 0.443
   2021-08: 0.443

Сгруппированные пики артистов (3):
   2021-02 (регион 2021-02..2021-03, max=0.993, пиков=2)
   2022-05 (регион 2022-05..2022-09, max=0.949, пиков=3)
   2024-02 (регион 2024-02, max=0.961, пиков=1)

Bootstrap устойчивости (топ-10):
2024-02    0.825
2021-03    0.810
2021-02    0.810
2022-05    0.805
2022-06    0.785
2024-04    0.600
2022-09    0.590
2024-01    0.530
2021-06    0.390
2025-07    0.320
dtype: float64



Продакшн-детектор (trimmed), stability:
2021-03    0.855
2021-02    0.845
2022-05    0.825
2024-02    0.825
2022-06    0.750
2022-09    0.580
dtype: float64


### Результат

> Числа в таблице взяты из продакшн-детектора (`cosine_identity.detect`, база `df_t`). Inline-расчёт в ячейке выше идёт на `df` без trim и даёт близкие, но не те же значения (например, 2021-02: 0.810 вместо 0.845; 2022-05: 0.805 вместо 0.825).

**Артисты дали 6 пиков** с высокой bootstrap-устойчивостью:

| Пик | Cosine distance | Bootstrap |
|---|---|---|
| 2021-02 | 0.993 | **84.5%** |
| 2021-03 | 0.958 | **85.5%** |
| 2022-05 | 0.949 | **82.5%** |
| 2022-06 | 0.933 | 75.0% |
| 2022-09 | 0.911 | 58.0% |
| 2024-02 | 0.961 | **82.5%** |

**Группировка ≤3 мес → 3 кластера:**
- `2021-02` — регион 2021-02..2021-03, max 0.993
- `2022-05` — регион 2022-05..2022-09, max 0.949
- `2024-02` — одна точка, max 0.961

**Жанры дали 15 пиков — это шум.** 2020-12, 2021-02, 03, 05, 06, 07, 08, 09, 2022-09, 2023-11, 12, 2024-01, 06, 07, 08 — 7 месяцев из 12 в 2021 году и вся вторая половина 2023–2024. Не точки, а фон. Отдельным слоем не берётся.

### 🔄 Поворот

**1. Cosine нашёл три чётких эпохи смены артистов.**

`2021-02`, `2022-05`, `2024-02` — все с bootstrap-устойчивостью ≥ 0.82 (продакшн-детектор).

**2. Жанры — не работают как отдельный слой.**

15 пиков размазаны по всей истории. Причина: у жанров высокая 
кардинальность — много категорий, каждая маленькая. Косинусное 
расстояние между любыми соседними месяцами всегда большое. Метод 
не различает «смена идентичности» vs «обычное разнообразие».

**3. Сравнение с наивным PELT пока невозможно.**

Наивный PELT из фазы 1 — смесь звука и поведения, его 7 точек 
гибридные. Единственное совпадение с Cosine — `2022-05`, но оно 
**отсеялось в bootstrap** (stability = 0.078). Других пересечений 
нет. Честное сравнение — в фазе 4, после разделения слоёв.

**4. Bootstrap нашёл зоны шире `find_peaks`.**

`2024-04` (0.60) и `2024-01` (0.53) — не пики по порогу, но 
bootstrap нашёл их сам. Зона вокруг `2024-02` шире, чем один месяц. 
То же с 2022: пик в мае + июнь + сентябрь — растянутая смена на 
4 месяца.

### → Ведёт к

Фаза 4: две проблемы, вскрытые за фазы 2–3.

- **`pen=10` пропускает и добавляет.** Bootstrap нашёл ~10 устойчивых 
  месяцев, которых не было в исходных 7. Нужен принципиальный штраф — 
  BIC-формула.
- **Разные слои видят разное.** Cosine ловит смену артистов при 
  похожем звучании. Смешивать в один PELT нельзя. Нужно разделить 
  на PELT-A (звук) и PELT-B (поведение) и сравнить три чистых слоя.

# Фаза 4. Разделение слоёв

## Фаза 4. Разделение слоёв

### Гипотеза

В фазе 1 наивный PELT смешал 4 звуковых признака и 3 поведенческих 
в один вход. Если звук и поведение — **один процесс**, разделение 
даст **те же точки**. Если **разные** — точки разойдутся, и это 
значит, что смешивать их в один PELT нельзя.

### Метод

Два независимых детектора из `src/detectors/`:

- **PELT-A** — 6 звуковых признаков (`valence`, `energy`, 
  `danceability`, `instrumentalness`, `acousticness`, `tempo`), 
  агрегация **median** по месяцу.
- **PELT-B** — 6 поведенческих (`hours`, `tracks`, `skip_rate`, 
  `played_frac`, `obsession_share`, `discovery_rate`), 4 из них 
  сглажены (rolling 3), агрегация **mean**.

Оба — с **BIC-штрафом** вместо `pen=10`. Плюс **bootstrap** 
с фильтром `stability ≥ 0.10`.

**Исключённые признаки:**
- `desktop_share` — один структурный скачок (Android → Windows), 
  не разладка. Давит остальные признаки в PELT.
- `orphan_share` — семантически близок к `discovery_rate`: оба 
  про «не моё / новое». Дублирование признака в модели.

In [6]:
# 4. Фаза 4. Разделение слоёв
# ============================================================================
# PELT-A на звуке + PELT-B на поведении. BIC-штраф вместо pen=10.
# detect() возвращает сырые точки. Фильтр stability >= 0.10 применяем явно.
# ============================================================================
import plotly.graph_objects as go
from temporal import trim_cold_start
from aggregators.music import build_monthly_sound, SOUND_FEATURES
from aggregators.behavior import build_monthly_behavior
from detectors.pelt_sound import detect as pelt_sound_detect
from detectors.pelt_behavior import detect as pelt_behavior_detect

STABILITY_THRESHOLD = 0.10

df_t = trim_cold_start(df)
print(f"После trim_cold_start: {df_t['period'].nunique()} месяцев, "
      f"{df_t['period'].min()} — {df_t['period'].max()}")

# --- PELT-A: звук ---
monthly_sound = build_monthly_sound(df_t)
res_a = pelt_sound_detect(monthly_sound)
stab_a = pd.Series(res_a['stability']).sort_values(ascending=False)

points_a_raw = list(res_a['points'])
points_a = [p for p in points_a_raw
            if stab_a.get(p, 0) >= STABILITY_THRESHOLD]

print("\n" + "=" * 70)
print("PELT-A (звук)")
print("=" * 70)
print(f"σ² = {res_a['sigma2']:.3f}, pen_BIC = {res_a['pen_bic']:.2f}")
print(f"\nСырые точки + bootstrap:")
for p in points_a_raw:
    s = stab_a.get(p, 0)
    mark = '✅' if s >= STABILITY_THRESHOLD else '⚠️ отсеяна'
    print(f"   {p}: {s:.3f} {mark}")
print(f"\nПосле фильтра ({STABILITY_THRESHOLD}): {points_a}")

# --- PELT-B: поведение ---
monthly_beh = build_monthly_behavior(df_t)
res_b = pelt_behavior_detect(monthly_beh)
stab_b = pd.Series(res_b['stability']).sort_values(ascending=False)

points_b_raw = list(res_b['points'])
points_b = [p for p in points_b_raw
            if stab_b.get(p, 0) >= STABILITY_THRESHOLD]

print("\n" + "=" * 70)
print("PELT-B (поведение)")
print("=" * 70)
print(f"σ² = {res_b['sigma2']:.3f}, pen_BIC = {res_b['pen_bic']:.2f}")
print(f"\nСырые точки + bootstrap:")
for p in points_b_raw:
    s = stab_b.get(p, 0)
    mark = '✅' if s >= STABILITY_THRESHOLD else '⚠️ отсеяна'
    print(f"   {p}: {s:.3f} {mark}")
print(f"\nПосле фильтра ({STABILITY_THRESHOLD}): {points_b}")

# --- Пересечения ---
common = set(points_a) & set(points_b)

print("\n" + "=" * 70)
print("СРАВНЕНИЕ СЛОЁВ")
print("=" * 70)
print(f"\nPELT-A ({len(points_a)}): {sorted(points_a)}")
print(f"PELT-B ({len(points_b)}): {sorted(points_b)}")
print(f"\nПересечений: {len(common)}")
if common:
    print(f"   Общие точки: {sorted(common)}")
else:
    print(f"   Нет общих точек — слои независимы")

# --- Визуализация ---
fig = go.Figure()

x_a = [pd.Period(p, 'M').to_timestamp() for p in points_a]
x_b = [pd.Period(p, 'M').to_timestamp() for p in points_b]

fig.add_trace(go.Scatter(
    x=x_a, y=[1] * len(x_a), mode='markers',
    marker=dict(size=14, color='#1f77b4', symbol='line-ns-open',
                line=dict(width=3)),
    name=f'PELT-A (звук, {len(points_a)})',
))
fig.add_trace(go.Scatter(
    x=x_b, y=[0] * len(x_b), mode='markers',
    marker=dict(size=14, color='#ff7f0e', symbol='line-ns-open',
                line=dict(width=3)),
    name=f'PELT-B (поведение, {len(points_b)})',
))

if common:
    x_c = [pd.Period(p, 'M').to_timestamp() for p in sorted(common)]
    fig.add_trace(go.Scatter(
        x=x_c, y=[0.5] * len(x_c), mode='markers',
        marker=dict(size=18, color='#d62728', symbol='diamond',
                    line=dict(width=2, color='black')),
        name=f'Пересечение ({len(common)})',
    ))

fig.update_layout(
    template='plotly_white',
    height=300,
    title='PELT-A vs PELT-B: точки разладки',
    xaxis_title='Месяц',
    yaxis=dict(tickmode='array', tickvals=[0, 0.5, 1],
               ticktext=['PELT-B', 'общие', 'PELT-A'],
               range=[-0.5, 1.5]),
    showlegend=True,
    margin=dict(l=80, r=40, t=60, b=40),
)
fig.show()

После trim_cold_start: 70 месяцев, 2020-10 — 2026-07

PELT-A (звук)
σ² = 0.420, pen_BIC = 10.71

Сырые точки + bootstrap:
   2022-11: 0.050 ⚠️ отсеяна
   2023-09: 0.240 ✅
   2024-02: 0.245 ✅
   2024-07: 0.160 ✅

После фильтра (0.1): ['2023-09', '2024-02', '2024-07']

PELT-B (поведение)
σ² = 0.339, pen_BIC = 8.64

Сырые точки + bootstrap:
   2021-08: 0.175 ✅
   2024-02: 0.135 ✅
   2024-12: 0.105 ✅
   2025-10: 0.150 ✅
   2026-03: 0.365 ✅

После фильтра (0.1): ['2021-08', '2024-02', '2024-12', '2025-10', '2026-03']

СРАВНЕНИЕ СЛОЁВ

PELT-A (3): ['2023-09', '2024-02', '2024-07']
PELT-B (5): ['2021-08', '2024-02', '2024-12', '2025-10', '2026-03']

Пересечений: 1
   Общие точки: ['2024-02']


### Результат

| Слой | Точки | σ² |
|---|---|---|
| PELT-A (звук) | `2023-09`, **`2024-02`**, `2024-07` | 0.420 |
| PELT-B (поведение) | `2021-08`, **`2024-02`**, `2024-12`, `2025-10`, `2026-03` | 0.339 |

**Bootstrap по точкам:**

PELT-A:
- `2023-09` — 0.240
- `2024-02` — 0.245
- `2024-07` — 0.160

PELT-B:
- `2021-08` — 0.175
- `2024-02` — 0.135
- `2024-12` — 0.105
- `2025-10` — 0.150
- `2026-03` — **0.365**

**Пересечение — одна точка: `2024-02`.**

### 🔄 Поворот

**1. Слои почти не пересекаются.**

Из 7 уникальных точек совпадает одна. Это не «разные шумные признаки 
одного процесса» — это **два независимых процесса**. Смешивать их 
в один PELT нельзя: признаки разной природы глушат друг друга.

**2. `2024-02` — особая точка.**

Bootstrap: PELT-A = 0.245, PELT-B = 0.135, Cosine = 0.825. 
`2024-02` — единственная дата, где профиль сдвинулся одновременно 
по звуку, поведению **и** артистам. Это самая обеспеченная 
доказательствами точка всей истории — три независимых слоя 
сошлись в одном месяце.

**3. `σ²` подтверждает: методы работают.**

- PELT-A: 0.420 — median устойчивее к выбросам (было 0.60 на mean).
- PELT-B: 0.339 — сглаживание 4 признаков убрало шум (было 0.72 
  на сырых).

**4. BIC-штраф вместо `pen=10`.**

- PELT-A: `pen_BIC = 10.71` — выведен из данных, не подобран.
- PELT-B: `pen_BIC = 8.64` — то же.

Это то, чего не хватало в фазе 1: штраф — не константа, а функция 
от σ², размерности и длины ряда.

### → Ведёт к

Фаза 5: раз слои независимы, «периодизация» — попытка найти 
стабильные эпохи между точками — вероятно, не подтвердится. 
История может оказаться не последовательностью глав, а непрерывным 
дрейфом. Проверка — явная.

# Фаза 5. Периодизация


## Фаза 5. Периодизация

### Гипотеза

Между точками разладки — **стабильные периоды**. Внутри периода 
характеристики не меняются. На границе — резкий переход. Если так, 
история делится на «главы»: эпоха A → эпоха B → эпоха C.

### Метод

Использованы **регионы cut-scan** как границы переходов. Между ними — 
6 «плато». Для каждого сравниваются медианы **первой и второй половины** 
периода по 4 звуковым признакам.

Если разница < 0.02 — плато однородно. Если больше — содержит скрытый 
сдвиг, cut-scan его не поймал.

**Ограничение метода.** Валидация на 4 признаках из 6 
(`valence`, `energy`, `danceability`, `instrumentalness`). 
`tempo` и `acousticness` исключены: у `tempo` диапазон 0–200, 
у `acousticness` — сильно скошенное распределение. Без Z-нормализации 
эти признаки задавили бы остальные.

Почему это не влияет на вывод: `max Δ` берётся как максимум по 
признакам. Добавление признаков только увеличивает max — не уменьшает. 
Если на 4 признаках плато нестабильно, на 6 оно тем более нестабильно 
при том же пороге. Результат устойчив к выбору набора.

In [7]:
# 5. Фаза 5. Периодизация
# ============================================================================
# Проверяем: между регионами нестабильности есть стабильные "плато"?
# Если да — история делится на главы. Если нет — непрерывный дрейф.
# Признаки: 4 (без tempo/acousticness), чтобы шкалы совпадали.
# ============================================================================
import pandas as pd
from aggregators.music import build_monthly_sound

VALIDATION_FEATURES = ['valence', 'energy', 'danceability', 'instrumentalness']

monthly_sound_full = build_monthly_sound(df_t)

# --- Cut-scan регионы (задаём явно — из отдельного прогона) ---
TRANSITION_ZONES = [
    ('2021-03', '2021-06'),
    ('2021-12', '2023-03'),
    ('2023-11', '2024-07'),
    ('2025-02', '2025-04'),
    ('2025-10', '2026-01'),
]

# --- Построить периоды между регионами ---
first_month = monthly_sound_full.index.min()
last_month = monthly_sound_full.index.max()

periods = []
cursor = first_month

for z_start, z_end in TRANSITION_ZONES:
    zs = pd.Period(z_start, 'M')
    ze = pd.Period(z_end, 'M')
    if cursor < zs:
        periods.append({'type': 'plateau', 'start': cursor, 'end': zs - 1})
    periods.append({'type': 'transition', 'start': zs, 'end': ze})
    cursor = ze + 1

if cursor <= last_month:
    periods.append({'type': 'plateau', 'start': cursor, 'end': last_month})

print(f"Всего отрезков: {len(periods)}")
print(f"  Плато:     {sum(1 for p in periods if p['type'] == 'plateau')}")
print(f"  Переходов: {sum(1 for p in periods if p['type'] == 'transition')}")

# --- Валидация плато ---
PRACTICAL_THRESHOLD = 0.02

print("\n" + "=" * 70)
print("ВАЛИДАЦИЯ: стабильны ли плато внутри?")
print("=" * 70)

results = []
for p in periods:
    if p['type'] != 'plateau':
        continue
    sub = monthly_sound_full[
        (monthly_sound_full.index >= p['start']) &
        (monthly_sound_full.index <= p['end'])
    ]
    if len(sub) < 4:
        continue

    mid = len(sub) // 2
    first_half = sub.iloc[:mid]
    second_half = sub.iloc[mid:]
    max_diff = 0
    for f in VALIDATION_FEATURES:
        diff = abs(first_half[f].median() - second_half[f].median())
        max_diff = max(max_diff, diff)

    is_stable = max_diff < PRACTICAL_THRESHOLD
    status = '✅ стабилен' if is_stable else '⚠️ неоднороден'

    print(f"\n{p['start']}..{p['end']} ({len(sub)} мес.) — {status}")
    print(f"   max Δ = {max_diff:.3f}")

    results.append({
        'period': f"{p['start']}..{p['end']}",
        'months': len(sub),
        'max_delta': max_diff,
        'is_stable': is_stable,
    })

# --- Сводка ---
df_plateaus = pd.DataFrame(results)

print("\n" + "=" * 70)
print("СВОДКА")
print("=" * 70)
n_stable = df_plateaus['is_stable'].sum()
n_total = len(df_plateaus)
print(f"Стабильных плато: {n_stable} из {n_total}")

if n_stable == n_total:
    print("✅ Периодизация работает — история делится на главы.")
elif n_stable >= n_total * 0.5:
    print("🟡 Большинство плато стабильны — периодизация частично работает.")
else:
    print("❌ Периодизация не работает — данные меняются непрерывно.")

Всего отрезков: 11
  Плато:     6
  Переходов: 5

ВАЛИДАЦИЯ: стабильны ли плато внутри?

2020-10..2021-02 (5 мес.) — ⚠️ неоднороден
   max Δ = 0.268

2021-07..2021-11 (5 мес.) — ⚠️ неоднороден
   max Δ = 0.175

2023-04..2023-10 (7 мес.) — ⚠️ неоднороден
   max Δ = 0.326

2024-08..2025-01 (6 мес.) — ⚠️ неоднороден
   max Δ = 0.031

2025-05..2025-09 (5 мес.) — ⚠️ неоднороден
   max Δ = 0.075

2026-02..2026-07 (6 мес.) — ⚠️ неоднороден
   max Δ = 0.044

СВОДКА
Стабильных плато: 0 из 6
❌ Периодизация не работает — данные меняются непрерывно.


## Результат

**0 из 6 плато стабильны.** Все содержат скрытый сдвиг внутри.

| Плато | Длительность | max Δ | Вердикт |
|---|---|---|---|
| 2020-10 .. 2021-02 | 5 мес | 0.268 | неоднородно |
| 2021-07 .. 2021-11 | 5 мес | 0.175 | неоднородно |
| 2023-04 .. 2023-10 | 7 мес | **0.326** | неоднородно |
| 2024-08 .. 2025-01 | 6 мес | 0.031 | неоднородно (близко к границе) |
| 2025-05 .. 2025-09 | 5 мес | 0.075 | неоднородно |
| 2026-02 .. 2026-07 | 6 мес | 0.044 | неоднородно |

Порог практической значимости: **0.02**.

## 🔄 Поворот

**1. Гипотеза «стабильные периоды между переходами» не подтвердилась.**

Между регионами cut-scan **нет** зон, где профиль замер. Данные 
меняются **непрерывно** — медленно, но без остановок.

**2. Особенно неоднородны три плато.**

- `2023-04..2023-10` — Δ = 0.326, в **16 раз** выше порога.
- `2020-10..2021-02` — Δ = 0.268.
- `2021-07..2021-11` — Δ = 0.175.

Внутри каждого — реальный сдвиг, который cut-scan не поймал.

**3. Близко к границе — только одно.**

`2024-08..2025-01` — Δ = 0.031. В 1.5 раза выше порога. Практически 
однородно, но формально не проходит. Единственный кандидат «почти плато».

**4. Устойчивость вывода к набору признаков.**

`max Δ` — это максимум по признакам. Добавление признаков **только 
увеличивает** max. Значит, если на 4 нестабильно, на 6 — тем более. 
Вывод «0 из 6» **не зависит** от выбора набора.

## → Ведёт к

Периодизация **не идёт в продакшн**. Это отрицательный результат, но 
важный: подтверждает, что история — **непрерывный дрейф**, а не 
последовательность глав.

Для дашборда следствие: **не рисуем «эпохи»**. Точки PELT/cosine 
остаются как точечные метки на непрерывной кривой — без обёртки 
«главы» или «периоды».

**Альтернатива периодизации** — штормовые эпизоды (rolling std). 
Не «стабильно / переход», а «где профиль бежит быстрее». Это **факт**, 
а не интерпретация. Разобрано в приложении B и фазе 12.

# Фаза 6. Self-percentile — структурный провал

## Фаза 6. Self-percentile — структурный провал

### Гипотеза

Раз периодизация не работает, нужен единый индекс изменчивости. 
Первая идея: для каждого месяца посчитать расстояние от предыдущего, 
затем разбить все расстояния на три уровня — Low / Medium / High — 
по терцилям **своего же ряда** (33-й и 67-й перцентили).

Подход работает без внешних данных. Так делают в большинстве 
дашбордов.

### Метод

Синтетика: два принципиально разных пользователя.

- **STABLE** — профиль не меняется, только шум наблюдений.
- **VARIABLE** — профиль сдвигается каждые 5 месяцев.

Симуляция: 48 месяцев, 150 событий в месяц, 3 признака. 
Сдвиг профиля — 0.6 стандартных отклонения.

Если self-percentile работает — STABLE получит почти всё Low, 
VARIABLE — почти всё High.

Для контраста — тот же тест через **null-based**: сравнение 
наблюдаемого сдвига с шумом, который дало бы случайное перемешивание 
реальных событий того же периода.

In [8]:
# 6. Фаза 6. Self-percentile — структурный провал
# ============================================================================
# Синтетика: два пользователя — стабильный и изменчивый.
# Self-percentile (терцили по собственной истории) обязан дать обоим
# 33/33/33 — по построению. Даже если один вообще не меняется.
# ============================================================================
import numpy as np
import pandas as pd

RNG_SEED = 7

def simulate_user(n_periods=48, events_per_period=150,
                  shift_periods=None, shift_size=0.0,
                  n_dims=3, noise=0.15, seed=42):
    """Симулирует события пользователя с опциональными сдвигами профиля."""
    rng_local = np.random.default_rng(seed)
    rows = []
    current_mean = np.zeros(n_dims)
    shift_periods = shift_periods or []

    for p in range(n_periods):
        if p in shift_periods:
            current_mean = current_mean + rng_local.normal(
                0, shift_size, size=n_dims
            )
        n_ev = rng_local.poisson(events_per_period)
        vals = rng_local.normal(current_mean, noise, size=(n_ev, n_dims))
        for row in vals:
            rows.append({'month': p, **{f'f{i}': row[i]
                                         for i in range(n_dims)}})
    return pd.DataFrame(rows)


# STABLE: профиль стоит на месте
df_stable = simulate_user(
    n_periods=48, events_per_period=150,
    shift_periods=[], shift_size=0.0, seed=1,
)

# VARIABLE: крупный сдвиг каждые 5 месяцев
df_variable = simulate_user(
    n_periods=48, events_per_period=150,
    shift_periods=list(range(5, 48, 5)), shift_size=0.6, seed=2,
)

print("=" * 70)
print("ВХОДНЫЕ ДАННЫЕ")
print("=" * 70)
print(f"STABLE:   {len(df_stable):,} событий, "
      f"{df_stable['month'].nunique()} месяцев")
print(f"VARIABLE: {len(df_variable):,} событий, "
      f"{df_variable['month'].nunique()} месяцев")


# --- Метод 1: self-percentile (провальный) ---
def self_percentile_levels(dist_series):
    """Терцили по собственному ряду. По определению даёт 33/33/33."""
    q33, q66 = dist_series.quantile([0.33, 0.66])
    labels = pd.Series('Medium', index=dist_series.index)
    labels[dist_series < q33] = 'Low'
    labels[dist_series > q66] = 'High'
    return labels


def cosine_distance(a, b):
    denom = np.linalg.norm(a) * np.linalg.norm(b)
    return float(1 - np.dot(a, b) / denom) if denom > 0 else np.nan


def compute_distances(df):
    """Расстояние между медианами соседних месяцев."""
    monthly = df.groupby('month')[['f0', 'f1', 'f2']].median().sort_index()
    vals = monthly.values
    out = []
    for i in range(1, len(vals)):
        a, b = vals[i - 1], vals[i]
        out.append(cosine_distance(a, b))
    return pd.Series(out, index=monthly.index[1:])


print("\n" + "=" * 70)
print("МЕТОД 1: SELF-PERCENTILE — результаты")
print("=" * 70)

for name, df_user in [('STABLE', df_stable), ('VARIABLE', df_variable)]:
    dist = compute_distances(df_user)
    labels = self_percentile_levels(dist)
    counts = labels.value_counts()
    print(f"\n{name}:")
    print(f"   Расстояния: median={dist.median():.4f}, "
          f"std={dist.std():.4f}")
    print(f"   Уровни: {dict(counts)}")


# --- Метод 2: null-based (правильный) ---
def null_based_levels(df_user, n_perm=200, p_high=95, seed=42):
    """Для каждой пары соседних месяцев — свой null. Уровень относительно null."""
    rng_local = np.random.default_rng(seed)
    monthly = df_user.groupby('month')[['f0', 'f1', 'f2']].median().sort_index()
    events = {m: df_user[df_user['month'] == m][['f0', 'f1', 'f2']].values
              for m in monthly.index}

    labels = {}
    for i in range(1, len(monthly.index)):
        m_from, m_to = monthly.index[i - 1], monthly.index[i]
        e_from, e_to = events[m_from], events[m_to]
        n_from, n_to = len(e_from), len(e_to)

        obs = cosine_distance(
            np.median(e_from, axis=0), np.median(e_to, axis=0),
        )

        pooled = np.vstack([e_from, e_to])
        nulls = np.empty(n_perm)
        for k in range(n_perm):
            perm = rng_local.permutation(len(pooled))
            a = pooled[perm[:n_from]]
            b = pooled[perm[n_from:n_from + n_to]]
            nulls[k] = cosine_distance(np.median(a, axis=0),
                                        np.median(b, axis=0))

        thr = np.percentile(nulls, p_high)
        labels[m_to] = 'High' if obs > thr else 'Low'

    return pd.Series(labels)


print("\n" + "=" * 70)
print("МЕТОД 2: NULL-BASED — результаты")
print("=" * 70)

for name, df_user in [('STABLE', df_stable), ('VARIABLE', df_variable)]:
    labels = null_based_levels(df_user, n_perm=100, seed=42)
    high_share = (labels == 'High').mean()
    n_high = (labels == 'High').sum()
    print(f"\n{name}:")
    print(f"   High-переходов: {n_high} / {len(labels)} ({high_share:.1%})")

# --- Визуализация: две панели ---
import plotly.graph_objects as go

# Собираем результаты обоих методов
self_pct = {}
null_pct = {}

for name, df_user in [('STABLE', df_stable), ('VARIABLE', df_variable)]:
    # self-percentile
    dist = compute_distances(df_user)
    labels_sp = self_percentile_levels(dist)
    self_pct[name] = (labels_sp == 'High').mean() * 100

    # null-based
    labels_nb = null_based_levels(df_user, n_perm=100, seed=42)
    null_pct[name] = (labels_nb == 'High').mean() * 100

users = ['STABLE', 'VARIABLE']
fig = go.Figure()
fig.add_trace(go.Bar(
    name='Self-percentile',
    x=users,
    y=[self_pct[u] for u in users],
    text=[f'{self_pct[u]:.1f}%' for u in users],
    textposition='outside',
))
fig.add_trace(go.Bar(
    name='Null-based (permutation)',
    x=users,
    y=[null_pct[u] for u in users],
    text=[f'{null_pct[u]:.1f}%' for u in users],
    textposition='outside',
))

fig.add_hline(y=33.3, line_dash='dash', line_color='gray',
              annotation_text='self-percentile всегда ~33%',
              annotation_position='top right')

fig.update_layout(
    template='plotly_white',
    height=400,
    title='Доля High-месяцев: self-percentile vs null-based',
    yaxis_title='High, %',
    barmode='group',
    margin=dict(l=60, r=40, t=80, b=40),
)
fig.show()    

ВХОДНЫЕ ДАННЫЕ
STABLE:   7,134 событий, 48 месяцев
VARIABLE: 7,199 событий, 48 месяцев

МЕТОД 1: SELF-PERCENTILE — результаты

STABLE:
   Расстояния: median=0.8446, std=0.5760
   Уровни: {'High': np.int64(16), 'Low': np.int64(16), 'Medium': np.int64(15)}

VARIABLE:
   Расстояния: median=0.0003, std=0.4081
   Уровни: {'High': np.int64(16), 'Low': np.int64(16), 'Medium': np.int64(15)}

МЕТОД 2: NULL-BASED — результаты

STABLE:
   High-переходов: 4 / 47 (8.5%)

VARIABLE:
   High-переходов: 15 / 47 (31.9%)


### Результат

**Self-percentile даёт обоим пользователям ровно 34% High-месяцев.** 
Одинаково — для стабильного и для изменчивого.

**Null-based различает их по существу:**

| Пользователь | Self-percentile | Null-based |
|---|---|---|
| STABLE | 34.0% | **8.5%** |
| VARIABLE | 34.0% | **31.9%** |

Разница в 3.7 раз между пользователями при null-based. 
Ноль разницы при self-percentile.

### 🔄 Поворот

**1. Self-percentile — структурный дефект, не проблема данных.**

Он **не может** различать пользователей в принципе. 33/33/33 — 
по определению терцилей. Это математика, не эмпирика. Метрика 
поймана до того, как попала в продакшн.

**2. Проблема в нулевой точке отсчёта.**

Self-percentile сравнивает месяц **с собственной медианой**. Если 
все расстояния малы — медиана тоже мала, и порог пропорционально 
мал. Уровень всегда 33/33/33.

**3. Внешний null решает проблему.**

Сравнение не с собой, а с **шумом, который дало бы случайное 
перемешивание реальных событий того же периода**. Это внешний 
ориентир — не зависит от того, насколько «громкий» сам ряд.

**4. Побочное наблюдение.**

`median` расстояний у STABLE (0.84) оказался больше, чем у VARIABLE 
(0.0003). Артефакт `cosine_distance` при нулевых векторах. 
На вывод не влияет — оба метода получили свои числа — но при 
прогоне на реальных данных этого нет: там векторы ненулевые.

### → Ведёт к

Фаза 7: если self-percentile не работает, нужен **permutation noise 
floor**. Сравнение наблюдаемого сдвига не с чужими данными и не с 
собственным рангом, а с нулём из случайного перемешивания. Плюс — 
двойной гейт (статистика + практика), иначе большие N делают 
что угодно «значимым».

# Фаза 7. Noise-floor через permutation


## Фаза 7. Noise-floor через permutation

### Гипотеза

Self-percentile не работает (фаза 6). Нужен **внешний** ориентир.
Для каждой пары соседних месяцев — перемешать события и посмотреть,
какое расстояние даст случайное разбиение. Если наблюдаемое
расстояние больше 99-го перцентиля null, сдвиг статистически
отличим от шума выборки. «Значимым» он считается только после
второго условия, практического гейта.

Плюс — **двойной гейт**:
- **Статистика:** observed > p99 null.
- **Практика:** хотя бы 2 признака сдвинулись выше «типичного»
  месячного сдвига (p75 по всей истории).

Оба условия обязательны. Иначе большие N делают что угодно «значимым».

### Метод

На **реальных данных**, не на синтетике. Признаки — 6 звуковых.

Для каждой пары соседних месяцев:
1. Считается Mahalanobis-расстояние между медианами (ковариация 
   разностей, Ledoit-Wolf shrinkage).
2. Null: 200 перемешиваний событий двух месяцев → распределение 
   расстояний.
3. Гейт: `observed > p99 null` **И** ≥2 признака выше p75.

Затем — rolling 12 → Variability как доля High-месяцев. 
`min_periods=12` — только полные окна.

База: **valid** (только осознанные прослушивания, скипы исключены).

In [9]:
# 7. Фаза 7. Noise-floor через permutation
# ============================================================================
# Финальная версия Variability: permutation noise floor + двойной гейт.
# Вызываем продакшн-модуль src/detectors/variability.py.
# ============================================================================
from detectors.variability import run_variability_pipeline

# valid-база: только осознанные прослушивания
df_valid = df_t[df_t['valid_play']].copy()

print(f"df_valid: {df_valid['period'].nunique()} месяцев, "
      f"{len(df_valid):,} событий "
      f"({len(df_valid)/len(df_t)*100:.1f}% от всех)")

monthly_valid = build_monthly_sound(df_valid)

# Прогон
R = run_variability_pipeline(
    df_valid,
    monthly_sound=monthly_valid,
    window=12,
    seed=42,
)

print("\n" + "=" * 70)
print("VARIABILITY — финальный результат")
print("=" * 70)
print(f"n_transitions:            {R['n_transitions']}")
print(f"n_valid_rolling_points:   {R['n_valid_rolling_points']}")
print(f"lifetime_score:           {R['lifetime_score']:.3f}")
print(f"current_score:            {R['current_score']:.3f}")
print(f"null_low / null_high:     {R['null_low']:.3f} / {R['null_high']:.3f}")
print(f"\nЭпохи высокого Variability:")
for e in R['epochs']:
    print(f"   {e['start']} … {e['end']}  "
          f"({e['duration_months']} мес)")

# Level по компасу 0.3/0.5
def level(s):
    if s < 0.3: return 'Low'
    if s <= 0.5: return 'Medium'
    return 'High'

print(f"\nLevel (0.3/0.5):")
print(f"   Lifetime: {level(R['lifetime_score'])}")
print(f"   Current:  {level(R['current_score'])}")

# --- Визуализация: score + null-коридор ---
import plotly.graph_objects as go

# Правильный ключ — rolling_series
var_series = R['rolling_series']

# rolling_series может быть pd.Series с index=Period, или с index=str.
# Приводим к единому формату.
if not isinstance(var_series, pd.Series):
    var_series = pd.Series(var_series)

if not isinstance(var_series.index, pd.PeriodIndex):
    try:
        var_series.index = pd.PeriodIndex(var_series.index, freq='M')
    except Exception:
        pass

x = var_series.index.to_timestamp()

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=x, y=var_series.values,
    mode='lines', name='Variability (rolling 12)',
    line=dict(width=2),
))

fig.add_hline(
    y=R['null_high'], line_dash='dash', line_color='red',
    annotation_text=f'граница шафла меток (95%) = {R["null_high"]:.3f}',
    annotation_position='top right',
)
fig.add_hline(
    y=R['null_low'], line_dash='dash', line_color='red',
    annotation_text=f'граница шафла меток (5%) = {R["null_low"]:.3f}',
    annotation_position='bottom right',
)

# Подсветка эпох
for ep in R['epochs']:
    fig.add_vrect(
        x0=pd.Period(ep['start'], 'M').to_timestamp(),
        x1=pd.Period(ep['end'], 'M').to_timestamp(),
        fillcolor='rgba(255, 100, 100, 0.10)',
        line_width=0, layer='below',
    )

fig.update_layout(
    template='plotly_white',
    height=400,
    title='Sound Variability (окно 12 мес) — заливка: эпохи High',
    xaxis_title='Месяц',
    yaxis_title='Variability',
    margin=dict(l=60, r=40, t=60, b=40),
)
fig.show()

df_valid: 70 месяцев, 45,706 событий (57.5% от всех)

VARIABILITY — финальный результат
n_transitions:            69
n_valid_rolling_points:   58
lifetime_score:           0.417
current_score:            0.083
null_low / null_high:     0.333 / 0.417

Эпохи высокого Variability:
   2021-10 … 2022-06  (9 мес)
   2024-01 … 2024-04  (4 мес)
   2024-06 … 2025-01  (8 мес)

Level (0.3/0.5):
   Lifetime: Medium
   Current:  Low


### Результат

**Lifetime Variability: 0.417** → Medium.
**Current (последнее окно rolling-12, т.е. 12 мес): 0.083** → Low.

Null-границы: **[0.333, 0.417]**. Lifetime = 0.417 — **на верхней 
границе**, ровно на пороге High.

**Три эпохи высокого Variability:**

| Эпоха | Длительность |
|---|---|
| 2021-10 .. 2022-06 | 9 мес |
| 2024-01 .. 2024-04 | 4 мес |
| 2024-06 .. 2025-01 | 8 мес |

Из 58 валидных точек rolling-12 — 21 попала в эпохи High (rolling > 0.5; сумма эпох 9 + 4 + 8).

### 🔄 Поворот

**1. Метод работает.**

Через `src/detectors/variability.py`:
- Mahalanobis distance между медианами соседних месяцев.
- Permutation noise floor — 200 итераций перемешивания.
- Двойной гейт: `observed > p99 null` **И** ≥ 2 признака выше p75 
  по всей истории.
- Rolling 12 → Variability как доля High-месяцев.

**2. Lifetime — ровно на границе High.**

0.417 совпадает с `null_high = 0.417`. Вариабельность пользователя 
статистически неотличима от случайной на верхней границе. 
Не «штормовой», не «штиль» — обычная динамика.

**3. Current — аномально тихо.**

0.083 — в **5 раз** ниже lifetime. За последние 12 месяцев профиль почти не менялся (один High-переход из 12). Это реальная стабильность, а не артефакт.

**4. Связь эпох с точками PELT не обнаружена.**

4 из 7 при ожидании 2.5, p = 0.22. `2024-02` совпадает со всеми
слоями, но это один момент времени — не подтверждение связи.

### → Ведёт к

Фаза 8: Variability получился для звука. Но есть **два независимых 
слоя** — звук и поведение. Логично посчитать Variability и для 
поведения. Проблема: как построить null для поведенческих признаков? 
Для звука перемешивались события внутри месяца. Для поведения — 
`skip_rate` и `played_frac` тавтологически связаны (корреляция −0.9). 
Нужен **свой null под структуру каждого признака**.

### Проверка чувствительности: статистический и практический пороги

Порог p99 выбран эвристически: при ~70 одновременных проверках он режет
ложные срабатывания, а практический гейт (≥2 признака выше p75) отсекает
шум дополнительно. Проверяем, насколько итоговая метрика зависит от обоих
порогов по отдельности.

**Часть 1 — статистический порог.** Меняется только `p_high` (90 / 95 / 99).
Практический порог p75 фиксирован.

**Часть 2 — практический порог.** Меняется только `percentile` (50…95).
Статистический порог p99 фиксирован.

Всё остальное не меняется: 6 признаков, valid-база, rolling-12,
`min_periods=12`, seed 42.

In [10]:
# 7b. Чувствительность Variability к порогам
# ============================================================================
# Две части:
#   1. Статистический порог p_high (90 / 95 / 99). Остальное фиксировано.
#   2. Практический порог percentile (50…95). Остальное фиксировано.
# ============================================================================
from detectors.variability_toolkit import VariabilityAnalyzer

events_7b = df_valid[['period'] + SOUND_FEATURES].dropna().copy()
events_7b['period_str'] = events_7b['period'].astype(str)


def _level(s):
    if s < 0.3: return 'Low'
    if s <= 0.5: return 'Medium'
    return 'High'


def _run(p_high, percentile):
    """Один прогон VariabilityAnalyzer с заданными порогами."""
    p75_local = (monthly_valid[SOUND_FEATURES].diff().abs()
                 .quantile(percentile / 100).to_dict())

    az = VariabilityAnalyzer(
        period_col='period_str',
        value_cols=SOUND_FEATURES,
        agg='median',
        n_boot=200, p_low=75, p_high=p_high,
        min_obs_per_period=5,
        practical_thresholds=p75_local,
        min_practical_features=2,
        random_state=42,
    ).fit(events_7b)

    sc = az.variability_score(window=12, n_boot=500, min_periods=12)
    tr = az.transitions_
    high = set(tr.index[tr['label'] == 'High'])
    stat_high = int((tr['statistical_label'] == 'High').sum())

    return {
        'p_high': p_high,
        'percentile': percentile,
        'стат. High (до гейта)': stat_high,
        'High после гейта': len(high),
        'lifetime': round(sc['lifetime_score'], 3),
        'current': round(sc['current_score'], 3),
        'level (0.3/0.5)': _level(sc['lifetime_score']),
        'high_set': high,
    }


# ---------- Часть 1: статистический порог ----------
print('=' * 70)
print('ЧАСТЬ 1. Статистический порог (practical = p75)')
print('=' * 70)

rows1, high_sets = [], {}
for p_high in (90, 95, 99):
    r = _run(p_high, percentile=75)
    high_sets[p_high] = r.pop('high_set')
    rows1.append(r)

sens1 = pd.DataFrame(rows1).drop(columns=['percentile'])
print(sens1.to_string(index=False))

print('\nПересечение High-месяцев с p99:')
for p in (90, 95):
    common = high_sets[p] & high_sets[99]
    print(f'   p{p} ∩ p99: {len(common)} из {len(high_sets[p])} (p{p}) / '
          f'{len(high_sets[99])} (p99)')


# ---------- Часть 2: практический порог ----------
print('\n' + '=' * 70)
print('ЧАСТЬ 2. Практический порог (p_high = 99)')
print('=' * 70)

rows2 = []
for pct in (50, 60, 70, 75, 80, 85, 90, 95):
    r = _run(p_high=99, percentile=pct)
    r.pop('high_set')
    rows2.append(r)

sens2 = pd.DataFrame(rows2).drop(columns=['p_high'])
print(sens2.to_string(index=False))

lifetimes = sens2['lifetime']
print(f'\nРазброс lifetime: {lifetimes.min():.3f} — {lifetimes.max():.3f}')
print(f'Уровни: {dict(sens2["level (0.3/0.5)"].value_counts())}')

ЧАСТЬ 1. Статистический порог (practical = p75)
 p_high  стат. High (до гейта)  High после гейта  lifetime  current level (0.3/0.5)
     90                     60                29     0.417    0.083          Medium
     95                     58                29     0.417    0.083          Medium
     99                     49                27     0.417    0.083          Medium

Пересечение High-месяцев с p99:
   p90 ∩ p99: 27 из 29 (p90) / 27 (p99)
   p95 ∩ p99: 27 из 29 (p95) / 27 (p99)

ЧАСТЬ 2. Практический порог (p_high = 99)
 percentile  стат. High (до гейта)  High после гейта  lifetime  current level (0.3/0.5)
         50                     49                43     0.667    0.417            High
         60                     49                36     0.583    0.167            High
         70                     49                32     0.542    0.083            High
         75                     49                27     0.417    0.083          Medium
         80         

### Результат

**Часть 1. Статистический порог (practical = p75):**

| p_high | Стат. High (до гейта) | High после гейта | Lifetime | Current | Level |
|---|---|---|---|---|---|
| 90 | 60 | 29 | 0.417 | 0.083 | Medium |
| 95 | 58 | 29 | 0.417 | 0.083 | Medium |
| 99 | 49 | 27 | 0.417 | 0.083 | Medium |

High-месяцы при p99 (27) целиком входят в наборы при p90 и p95 (29 каждый); разница — два месяца.

**Часть 2. Практический порог (p_high = 99):**

| percentile | High после гейта | Lifetime | Current | Level |
|---|---|---|---|---|
| 50 | 43 | 0.667 | 0.417 | High |
| 60 | 36 | 0.583 | 0.167 | High |
| 70 | 32 | 0.542 | 0.083 | High |
| **75** | **27** | **0.417** | **0.083** | **Medium** |
| 80 | 21 | 0.333 | 0.000 | Medium |
| 85 | 16 | 0.250 | 0.000 | Low |
| 90 | 10 | 0.125 | 0.000 | Low |
| 95 | 8 | 0.083 | 0.000 | Low |

Разброс lifetime: **0.083 — 0.667**. Уровень принимает все три значения:
High (p50–p70), Medium (p75–p80), Low (p85–p95). Current падает до нуля
начиная с p80.

### 🔄 Поворот

**1. Итоговая метрика не зависит от статистического порога.** Lifetime, current и уровень одинаковы на p90, p95 и p99.

**2. Основную фильтрацию делает практический гейт.** Статистический порог меняет число «значимых» переходов с 60 до 49, но после практического гейта остаётся 29 и 27. Именно гейт (≥2 признака выше p75) отсекает малые сдвиги, а p99 лишь немного его дополняет.

**3. Что это не доказывает.** Проверены три порога на одном seed. Практический порог p75 здесь не варьировался и остаётся эвристикой, откалиброванной на этих данных.

**4. Итоговая метрика полностью определяется практическим порогом.**
При смене квантиля от p50 до p95 lifetime меняется в восемь раз,
уровень переключается трижды. Практический порог — единственный
параметр, от которого зависит результат.

**5. Это свойство метрики, а не дефект реализации.** Практический
порог — это p75 собственных сдвигов. По построению доля месяцев выше
него всегда ~25% при любом масштабе. При p50 — ~50%, при p95 — ~5%.
Score механически следует за выбранным квантилем. Permutation-гейт
при этом фиксирован (p99) и почти не фильтрует — в таблице
`High после гейта` совпадает с числом месяцев выше практического
порога.

**6. Что метрика измеряет.** Не силу изменений. Сгруппированность
перемен во времени: сколько месяцев были нетипично крупными
относительно собственного фона. Это осмысленная характеристика,
но не та, как её иногда читают.

**7. Что это не доказывает.** Проверена чувствительность к двум
порогам отдельно. Альтернативная метрика без порогов (Changiness —
медиана Mahalanobis-сдвигов) в продакшн не вошла; её единицы
self-relative, уровень не калиброван. Эпохи Sound Variability
(2021-10…2022-06, 2024-01…2024-04, 2024-06…2025-01) требуют
отдельной проверки на артефакт порядка High-меток — не сделана.

# Фаза 8. Разные нули для Sound и Behavior

## Фаза 8. Разные нули для Sound и Behavior

### Гипотеза

Sound Variability работает (фаза 7). Логично посчитать **Behavior 
Variability** тем же методом. Но применить его «в лоб» нельзя:

- **Sound** — event-level. У каждого трека есть аудио-признаки. 
  Null: перемешивание событий внутри двух соседних месяцев.
- **Behavior** — mix. `hours` и `tracks` — event-level, но 
  `obsession_share` — session-level (свойство сессии, не события).

Один null на всё не подойдёт. Нужен **свой под структуру каждого 
признака**.

### Метод

Для каждого поведенческого признака — свой нуль:

| Признак | Уровень | Null |
|---|---|---|
| `hours` | event | pool событий → shuffle → пересчёт |
| `tracks` | event | то же |
| `skip_rate` | event | то же |
| `played_frac` | event | то же |
| `obsession_share` | **session** | pool сессий → shuffle → пересчёт |

Плюс — **кластерное голосование** вместо признакового. `skip_rate` 
и `played_frac` коррелируют (−0.9): два измерения одного явления 
«дослушал / не дослушал». Считаются одним голосом.

Кластеры:
- **volume**: `hours`, `tracks`
- **completion**: `skip_rate`, `played_frac`
- **obsession**: `obsession_share`

High-месяц — если **≥ 2 кластеров** дали High. Практический порог — 
p75 сырых месячных сдвигов по каждому признаку.

In [11]:
# 8. Фаза 8. Разные нули для Sound и Behavior
# ============================================================================
# Behavior Variability: свой null под структуру каждого признака.
# Event-level признаки — permutation событий.
# Session-level (obsession) — permutation сессий.
# Кластерное голосование: skip_rate и played_frac считаются одним голосом
# (они коррелируют -0.9 — это два измерения одного явления).
# Sound Variability берём из фазы 7 (rolling_series), не пересчитываем.
# ============================================================================
import numpy as np
import pandas as pd
import plotly.graph_objects as go

N_BOOT_PERM = 200
MIN_EVENTS = 5
BEH_CLUSTERS = {
    'volume':     ['hours', 'tracks'],
    'completion': ['skip_rate', 'played_frac'],
    'obsession':  ['obsession_share'],
}

# --- Калибровка практических порогов (p75 сырых месячных сдвигов) ---
mb_calib = df_t.groupby('period').agg(
    hours=('ms_played', lambda x: x.sum() / 3_600_000),
    tracks=('spotify_track_uri', 'nunique'),
    skip_rate=('is_skip', 'mean'),
    played_frac=('played_frac', 'mean'),
).sort_index()

def build_session_obsession_flag(df_e, gap=30, cnt_thr=3, share_thr=0.10):
    s = df_e.sort_values(['account', 'ts_naive']).copy()
    s['prev_ts'] = s.groupby('account')['ts_naive'].shift(1)
    s['gap'] = (s['ts_naive'] - s['prev_ts']).dt.total_seconds() / 60
    s['new_sess'] = s['gap'].isna() | (s['gap'] > gap)
    s['session_id'] = s.groupby('account')['new_sess'].cumsum()

    pair = (s.groupby(['account', 'session_id', 'spotify_track_uri'])
              .size().rename('cnt').reset_index())
    total = (s.groupby(['account', 'session_id'])
               .size().rename('n_ev').reset_index())
    pair = pair.merge(total, on=['account', 'session_id'])
    pair['is_pair'] = (pair['cnt'] >= cnt_thr) & \
                      (pair['cnt'] / pair['n_ev'] >= share_thr)

    sess = (pair.groupby(['account', 'session_id'])['is_pair']
                 .any().rename('is_obsession').reset_index())
    sess['period'] = (s.groupby(['account', 'session_id'])['period']
                       .first().values)
    return sess

sess_flag = build_session_obsession_flag(df_t)
mb_calib['obsession_share'] = (sess_flag.groupby('period')['is_obsession']
                                .mean()
                                .reindex(mb_calib.index).fillna(0.0))

CAL = {}
for f in ['hours', 'tracks', 'skip_rate', 'played_frac', 'obsession_share']:
    CAL[f] = mb_calib[f].diff().abs().quantile(0.75)

print("Практические пороги (p75):")
for k, v in CAL.items():
    print(f"   {k:18s}: {v:.4f}")

def gate(obs, nulls, thr_prac, p_low=75, p_high=99):
    thr_low = np.percentile(nulls, p_low)
    thr_high = np.percentile(nulls, p_high)
    if obs > thr_high and obs >= thr_prac:
        return 'High'
    if obs > thr_low and obs >= thr_prac:
        return 'Medium'
    return 'Low'

def null_perm(a, b, fn, na, nb, n_boot, rng):
    pool = pd.concat([a, b], ignore_index=True)
    out = np.empty(n_boot)
    for k in range(n_boot):
        p = rng.permutation(len(pool))
        out[k] = abs(fn(pool.iloc[p[na:na + nb]]) -
                     fn(pool.iloc[p[:na]]))
    return out

agg_hours       = lambda e: e['ms_played'].sum() / 3_600_000
agg_tracks      = lambda e: e['spotify_track_uri'].nunique()
agg_skip_rate   = lambda e: e['is_skip'].mean()
agg_played_frac = lambda e: e['played_frac'].mean()
agg_obsession   = lambda s: s['is_obsession'].mean()

# --- Прогон по месяцам ---
rng = np.random.default_rng(42)
months = sorted(df_t['period'].dropna().unique())
events = df_t[['account', 'ts_naive', 'period', 'ms_played',
               'spotify_track_uri', 'is_skip', 'played_frac']].copy()

per_month = {}
for i in range(len(months) - 1):
    m, m1 = months[i], months[i + 1]
    e0 = events[events['period'] == m]
    e1 = events[events['period'] == m1]
    if len(e0) < MIN_EVENTS or len(e1) < MIN_EVENTS:
        continue

    row = {}
    for name, agg in [('hours', agg_hours), ('tracks', agg_tracks),
                      ('skip_rate', agg_skip_rate),
                      ('played_frac', agg_played_frac)]:
        obs = abs(agg(e1) - agg(e0))
        nulls = null_perm(e0, e1, agg, len(e0), len(e1),
                          N_BOOT_PERM, rng)
        row[name] = gate(obs, nulls, CAL[name])

    s0 = sess_flag[sess_flag['period'] == m]
    s1 = sess_flag[sess_flag['period'] == m1]
    if len(s0) >= 3 and len(s1) >= 3:
        obs = abs(agg_obsession(s1) - agg_obsession(s0))
        nulls = null_perm(s0, s1, agg_obsession, len(s0), len(s1),
                          N_BOOT_PERM, rng)
        row['obsession_share'] = gate(obs, nulls, CAL['obsession_share'])

    if row:
        per_month[str(m1)] = row

# --- Кластерное голосование ---
events_beh = pd.Series(0, index=pd.PeriodIndex(
    [str(m) for m in months[1:]], freq='M'))
for month_str, per_feat in per_month.items():
    cluster_high = 0
    for cname, feats in BEH_CLUSTERS.items():
        labs = [per_feat[f] for f in feats if f in per_feat]
        if 'High' in labs:
            cluster_high += 1
    if cluster_high >= 2:
        events_beh[pd.Period(month_str, 'M')] = 1

n_high_beh = int(events_beh.sum())
print(f"\nBehavior High-месяцев: {n_high_beh} из {len(events_beh)} "
      f"({events_beh.mean():.3f})")

# --- Behavior Variability (rolling 12) ---
var_beh = events_beh.rolling(12, min_periods=4).mean()
lifetime_beh = var_beh.median()

# --- Sound Variability — берём из фазы 7, НЕ пересчитываем ---
var_sound = R['rolling_series']
if not isinstance(var_sound, pd.Series):
    var_sound = pd.Series(var_sound)
if not isinstance(var_sound.index, pd.PeriodIndex):
    var_sound.index = pd.PeriodIndex(var_sound.index, freq='M')

lifetime_sound = var_sound.median()

print(f"\nBehavior Variability:")
print(f"   Lifetime:  {lifetime_beh:.3f}")
print(f"   Current:   {var_beh.iloc[-6:].mean():.3f}")

print(f"\nSound Variability:")
print(f"   Lifetime:  {lifetime_sound:.3f}")

# --- Сырая корреляция ---
common = var_sound.dropna().index.intersection(var_beh.dropna().index)
r_raw = np.corrcoef(var_sound[common], var_beh[common])[0, 1]
print(f"\ncorr(Sound, Behavior) сырая: {r_raw:.3f}")

# --- Визуализация ---
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=var_sound.dropna().index.to_timestamp(),
    y=var_sound.dropna().values,
    mode='lines', name='Sound Variability',
    line=dict(width=2),
))
fig.add_trace(go.Scatter(
    x=var_beh.dropna().index.to_timestamp(),
    y=var_beh.dropna().values,
    mode='lines', name='Behavior Variability',
    line=dict(width=2),
))
fig.update_layout(
    template='plotly_white',
    height=400,
    title='Sound vs Behavior Variability (окно 12 мес)',
    xaxis_title='Месяц',
    yaxis_title='Variability',
    margin=dict(l=60, r=40, t=60, b=40),
)
fig.show()
var_beh_12 = events_beh.rolling(12, min_periods=12).mean()
print(f"\nBehavior lifetime, min_periods=12: {var_beh_12.median():.3f} (n={var_beh_12.notna().sum()})")

Практические пороги (p75):
   hours             : 39.2928
   tracks            : 447.0000
   skip_rate         : 0.2166
   played_frac       : 0.2283
   obsession_share   : 0.0805

Behavior High-месяцев: 10 из 69 (0.145)

Behavior Variability:
   Lifetime:  0.167
   Current:   0.278

Sound Variability:
   Lifetime:  0.417

corr(Sound, Behavior) сырая: -0.272



Behavior lifetime, min_periods=12: 0.125 (n=58)


### Результат

**Behavior Variability:**
- Lifetime: **0.167** (на полных окнах `min_periods=12` — 0.125; Sound: 0.417)
- Current: 0.278
- High-месяцев: **10 из 69**

**Sound Variability (из фазы 7):** 0.417 Lifetime.

**Сырая корреляция Sound ↔ Behavior: −0.272.**

Практические пороги (p75 сырых сдвигов):

| Признак | Порог |
|---|---|
| hours | 39.29 |
| tracks | 447.0 |
| skip_rate | 0.217 |
| played_frac | 0.228 |
| obsession_share | 0.081 |

### 🔄 Поворот

**1. Один null на всё не работает.**

Sound — event-level: у каждого трека есть аудио-признаки. 
Перемешивание событий внутри двух месяцев даёт null.

Behavior — mix: `hours` и `tracks` event-level, но `obsession_share` 
— session-level. Перемешать события для obsession_share нельзя — 
это сломает структуру.

Решение — **свой null под структуру каждого признака**:
- event-level → permutation событий;
- session-level → permutation сессий.

**2. Двойное голосование — скрытая ошибка.**

`skip_rate` и `played_frac` коррелируют **−0.9**: «сколько скипнул» 
и «сколько дослушал» — два измерения одного явления. При подсчёте 
как двух независимых голосов они **удваивают** один сигнал.

**Кластерное голосование решает:**
- volume: hours, tracks
- completion: skip_rate, played_frac
- obsession: obsession_share

High-месяц — если ≥ 2 кластеров дали High.

**3. Behavior Variability ниже Sound, и это не артефакт неполных окон.**

На полных окнах (`min_periods=12`, как у Sound) Behavior lifetime = 0.125, с неполными (`min_periods=4`) — 0.167; Sound — 0.417. Правило High у Behavior строже (≥2 кластера, у каждого свой p99 и практический порог), поэтому часть разницы может быть эффектом правила, а не поведения. Вывод «поведение меняется реже, чем вкус» — предварительный.

**4. Сырая корреляция −0.272.**

Отрицательная. Похоже на противофазу: где звук бежит, поведение 
стоит. Или наоборот.

**Но это ещё не вывод.** Корреляция между временными рядами может 
быть **артефактом общего тренда**, а не функциональной связью. 
Обе Variability могут независимо падать со временем — и дать 
отрицательную корреляцию без всякой связи.

Проверка — в фазе 9.

### → Ведёт к

Фаза 9: **детренд**. Убирается линейный тренд из обоих рядов, 
корреляция пересчитывается. Если −0.272 остаётся — связь реальна. 
Если падает до нуля — противофаза была иллюзией.

# Фаза 9. Detrend — не установленные связи

## Фаза 9. Detrend — не установленные связи

### Гипотеза

Сырая корреляция Sound ↔ Behavior = −0.272. Два ряда могут быть 
**связаны** (противофаза: где один бежит, другой стоит). А могут 
**независимо дрейфовать** — Sound падает со временем, Behavior 
растёт. Тогда корреляция будет отрицательной **без всякой связи**.

Стандарт для временных рядов — **детренд**: убрать линейный тренд 
из обоих рядов, пересчитать корреляцию.

### Метод

1. Для каждого ряда — линейный тренд `y = a + b·t`.
2. Вычитание тренда: `y_detr = y − trend`.
3. Корреляция между детрендированными рядами.
4. Проверка по **всем парам осей**, а не только Sound ↔ Behavior — 
   чтобы поймать другие фантомные связи.

Пары для проверки: Sound Variability, Behavior Variability, 
Discovery, Retention, Burnout. 10 пар.

In [12]:
# 9. Фаза 9. Detrend — не установленные связи
# ============================================================================
# Убираем линейный тренд из всех временных рядов. Пересчитываем корреляции.
# Если p_raw значимо, а p_detr нет — связь НЕ УСТАНОВЛЕНА (тренд объясняет
# основную часть), не «доказанно ложная». Мощность теста при n≈58 мала.
# ============================================================================
import numpy as np
import pandas as pd
from scipy.stats import pearsonr


def detrend(series):
    """Убирает линейный тренд."""
    x = series.values
    t = np.arange(len(x))
    trend = np.polyfit(t, x, 1)
    return pd.Series(x - np.polyval(trend, t), index=series.index)


def compare_corr(a, b, name):
    """Сырая vs детрендированная корреляция."""
    common = a.dropna().index.intersection(b.dropna().index)
    if len(common) < 10:
        print(f"\n{name}: мало данных (n={len(common)})")
        return None

    aa = a.loc[common]
    bb = b.loc[common]

    r_raw, p_raw = pearsonr(aa, bb)
    aa_d = detrend(aa)
    bb_d = detrend(bb)
    r_detr, p_detr = pearsonr(aa_d, bb_d)

    print(f"\n{name} (n={len(common)}):")
    print(f"   Сырая:   r = {r_raw:+.3f}, p = {p_raw:.4f}")
    print(f"   Детренд: r = {r_detr:+.3f}, p = {p_detr:.4f}")

    if p_raw < 0.05 and p_detr >= 0.05:
        verdict = "🟡 не установлена — тренд объясняет основную часть"
    elif p_detr < 0.05 and abs(r_detr) > abs(r_raw) * 0.7:
        verdict = "✅ функциональная"
    elif p_detr < 0.05:
        verdict = "🟡 частично функциональная"
    else:
        verdict = "⚪ нет связи"

    print(f"   → {verdict}")

    return {'name': name, 'r_raw': r_raw, 'p_raw': p_raw,
            'r_detr': r_detr, 'p_detr': p_detr, 'verdict': verdict}


# --- Пары для проверки ---
print("=" * 70)
print("DETREND: ВСЕ КОРРЕЛЯЦИИ")
print("=" * 70)

results = []

# Sound ↔ Behavior
results.append(compare_corr(
    var_sound, var_beh, "Sound Var ↔ Behavior Var"
))

# --- Помесячные Discovery / Retention / Burnout ---

# Discovery помесячно
first_seen = (df_t.groupby('spotify_track_uri')['ts_naive'].min()
              .dt.to_period('M'))
df_t_copy = df_t.copy()
df_t_copy['_first_month'] = df_t_copy['spotify_track_uri'].map(first_seen)
df_t_copy['_is_discovery'] = df_t_copy['period'] == df_t_copy['_first_month']
discovery_monthly = (df_t_copy[df_t_copy['valid_play']]
                     .groupby('period')['_is_discovery']
                     .mean()
                     .sort_index())
discovery_monthly.index = pd.PeriodIndex(discovery_monthly.index, freq='M')

# Retention (когортный 90d)
valid_ev = df_t_copy[df_t_copy['valid_play']].copy()
valid_ev['first_month'] = valid_ev['first_valid_seen'].dt.to_period('M')
first_events = valid_ev[valid_ev['ts_naive'] == valid_ev['first_valid_seen']]
repeats = valid_ev[valid_ev['ts_naive'] > valid_ev['first_valid_seen']].copy()
repeats['within_90'] = (
    repeats['ts_naive'] <= repeats['first_valid_seen'] + pd.Timedelta(days=90)
)
returned_90 = set(repeats[repeats['within_90']]['spotify_track_uri'].unique())

first_events = first_events.copy()
first_events['returned'] = first_events['spotify_track_uri'].isin(returned_90)
first_events['cohort'] = first_events['first_valid_seen'].dt.to_period('M')

retention_monthly = first_events.groupby('cohort')['returned'].mean()
retention_monthly.index = pd.PeriodIndex(retention_monthly.index, freq='M')

# Burnout (rolling 12)
track_counts = valid_ev.groupby('spotify_track_uri').size()
core = track_counts[track_counts >= 5].index
last_seen = (valid_ev[valid_ev['spotify_track_uri'].isin(core)]
             .groupby('spotify_track_uri')['ts_naive'].max())

burnout_vals = {}
for m in sorted(valid_ev['period'].unique()):
    m_end = m.to_timestamp(how='end')
    cutoff = m_end - pd.Timedelta(days=365)
    first_ts = (valid_ev[valid_ev['spotify_track_uri'].isin(core)]
                .groupby('spotify_track_uri')['ts_naive'].min())
    eligible = first_ts[first_ts <= cutoff].index
    if len(eligible) == 0:
        burnout_vals[m] = np.nan
        continue
    burnout_vals[m] = (last_seen[eligible] < cutoff).mean()
burnout_monthly = pd.Series(burnout_vals).sort_index()
burnout_monthly.index = pd.PeriodIndex(burnout_monthly.index, freq='M')

# --- Все пары ---
print("\n" + "=" * 70)
print("ОСИ МЕЖДУ СОБОЙ")
print("=" * 70)
compare_corr(discovery_monthly, retention_monthly, "Discovery ↔ Retention")
compare_corr(discovery_monthly, burnout_monthly,   "Discovery ↔ Burnout")
compare_corr(retention_monthly, burnout_monthly,   "Retention ↔ Burnout")

print("\n" + "=" * 70)
print("SOUND/BEHAVIOR VARIABILITY ↔ ОСИ")
print("=" * 70)
compare_corr(var_sound, discovery_monthly, "Sound Var ↔ Discovery")
compare_corr(var_sound, retention_monthly, "Sound Var ↔ Retention")
compare_corr(var_sound, burnout_monthly,   "Sound Var ↔ Burnout")
compare_corr(var_beh,   discovery_monthly, "Behavior Var ↔ Discovery")
compare_corr(var_beh,   retention_monthly, "Behavior Var ↔ Retention")
compare_corr(var_beh,   burnout_monthly,   "Behavior Var ↔ Burnout")


# --- Сводка ---
print("\n" + "=" * 70)
print("СВОДКА: ЧТО НЕ УСТАНОВЛЕНО / ЧТО ФУНКЦИОНАЛЬНО")
print("=" * 70)

all_pairs = [
    ('Sound Var ↔ Behavior Var', var_sound, var_beh),
    ('Sound Var ↔ Discovery',    var_sound, discovery_monthly),
    ('Sound Var ↔ Retention',    var_sound, retention_monthly),
    ('Sound Var ↔ Burnout',      var_sound, burnout_monthly),
    ('Behavior Var ↔ Discovery', var_beh,   discovery_monthly),
    ('Behavior Var ↔ Retention', var_beh,   retention_monthly),
    ('Behavior Var ↔ Burnout',   var_beh,   burnout_monthly),
    ('Discovery ↔ Retention',    discovery_monthly, retention_monthly),
    ('Discovery ↔ Burnout',      discovery_monthly, burnout_monthly),
    ('Retention ↔ Burnout',      retention_monthly, burnout_monthly),
]

print(f"\n{'Пара':<30} {'r_raw':<10} {'r_detr':<10} {'Вердикт'}")
print("-" * 75)

n_unsettled = 0
n_real = 0
for name, a, b in all_pairs:
    common = a.dropna().index.intersection(b.dropna().index)
    if len(common) < 10:
        continue
    aa = a.loc[common]; bb = b.loc[common]
    r_raw, p_raw = pearsonr(aa, bb)
    aa_d = detrend(aa); bb_d = detrend(bb)
    r_detr, p_detr = pearsonr(aa_d, bb_d)

    if p_raw < 0.05 and p_detr >= 0.05:
        v = "🟡 не установлена"
        n_unsettled += 1
    elif p_detr < 0.05:
        v = "✅ функц."
        n_real += 1
    elif p_raw < 0.05:
        v = "🟡 часть"
    else:
        v = "⚪ нет"

    print(f"{name:<30} {r_raw:+.3f}     {r_detr:+.3f}     {v}")

print(f"\nНе установлено: {n_unsettled}, функциональных: {n_real}")

DETREND: ВСЕ КОРРЕЛЯЦИИ

Sound Var ↔ Behavior Var (n=58):
   Сырая:   r = -0.272, p = 0.0389
   Детренд: r = +0.243, p = 0.0665
   → 🟡 не установлена — тренд объясняет основную часть

ОСИ МЕЖДУ СОБОЙ

Discovery ↔ Retention (n=70):
   Сырая:   r = +0.094, p = 0.4384
   Детренд: r = +0.112, p = 0.3551
   → ⚪ нет связи

Discovery ↔ Burnout (n=58):
   Сырая:   r = -0.174, p = 0.1926
   Детренд: r = -0.033, p = 0.8076
   → ⚪ нет связи

Retention ↔ Burnout (n=58):
   Сырая:   r = -0.132, p = 0.3228
   Детренд: r = +0.015, p = 0.9131
   → ⚪ нет связи

SOUND/BEHAVIOR VARIABILITY ↔ ОСИ

Sound Var ↔ Discovery (n=58):
   Сырая:   r = +0.021, p = 0.8727
   Детренд: r = -0.091, p = 0.4958
   → ⚪ нет связи

Sound Var ↔ Retention (n=58):
   Сырая:   r = +0.168, p = 0.2082
   Детренд: r = +0.099, p = 0.4607
   → ⚪ нет связи

Sound Var ↔ Burnout (n=58):
   Сырая:   r = -0.553, p = 0.0000
   Детренд: r = -0.201, p = 0.1313
   → 🟡 не установлена — тренд объясняет основную часть

Behavior Var ↔ Discovery 

### Результат

Проверены **10 пар** осей и Variability на артефакт тренда.

| Пара | r_raw | r_detr | Вердикт |
|---|---|---|---|
| Sound Var ↔ Behavior Var | −0.272 | +0.243 |  🟡 не установленат |
| **Sound Var ↔ Burnout** | **−0.553** | **−0.201** | ** 🟡 не установлена** |
| **Behavior Var ↔ Burnout** | +0.421 | **−0.757** | ✅ функциональная |
| Sound Var ↔ Discovery | +0.021 | −0.091 | ⚪ нет |
| Sound Var ↔ Retention | +0.168 | +0.099 | ⚪ нет |
| Behavior Var ↔ Discovery | −0.094 | +0.059 | ⚪ нет |
| Behavior Var ↔ Retention | −0.168 | −0.184 | ⚪ нет |
| Discovery ↔ Retention | +0.094 | +0.112 | ⚪ нет |
| Discovery ↔ Burnout | −0.174 | −0.033 | ⚪ нет |
| Retention ↔ Burnout | −0.132 | +0.015 | ⚪ нет |

**Итог:** 2 не установлены, 1 функциональная, 7 без связи.

Оговорка: −0.757 на 58 точках. Behavior Variability разрежен (10 High из 69). Проверка leave-one-out (фаза 10): при удалении любого из 10 High-месяцев r остаётся в диапазоне −0.614…−0.842 (медиана −0.703), знак не меняется. Связь не держится на одном-двух месяцах. Не проверялось: p-value при перекрывающихся окнах (rolling-12 и годовое окно Burnout) оптимистичен, эффективный n меньше 58.

### 🔄 Поворот

**1. «Sound ↔ Burnout = −0.553» — не установлена.**

Самая сильная корреляция в исследовании. `p < 0.0001`. После 
детренда — `r = −0.201, p = 0.13`. Основная часть корреляции 
объясняется общим трендом двух рядов: Sound Variability падает 
(вкус стабилизировался), Burnout растёт (старые core-треки 
перешагнули 12-мес порог).

Но это **не доказательство отсутствия связи**. При n = 58 мощность 
теста для эффекта масштаба r ≈ 0.20 — около 30–35%. Тест не 
различает «связи нет» и «связь есть, но слабая». Корректный 
статус — **не установлена**, не **ложная**.

**2. «Sound ↔ Behavior Var = −0.272» — тоже не установлена.**

Сырая `p = 0.0389`, после детренда `r = +0.243, p = 0.0665`. 
Знак перевернулся — сильная противофаза не подтвердилась. Но 
`p = 0.0665` — **на границе чувствительности теста**, а не 
«далеко в шуме». Поэтому статус тот же: **не установлена**.

**3. Behavior Var ↔ Burnout — единственная функциональная.**

Сырая `+0.421` → детренд `−0.757`. Знак перевернулся, **связь 
усилилась**. Это редкий случай, когда общий тренд **маскировал** 
реальную связь.

**4. Detrend обязателен для временных рядов.**

Стандарт эконометрики, который изначально был пропущен. 
Два сильных сигнала (`−0.553` и `−0.272`) не подтвердились после 
детренда — цена этого пропуска.

**5. Компас функционально 4-мерный.**

Все 6 пар **внутри компаса** (Discovery, Retention, Burnout, 
Sound Var) после детренда — не установлены. Оси независимы.

**Behavior Variability не в компасе** — и правильно. Она связана 
с Burnout. Если бы вошла как отдельная ось, компас перестал бы 
быть 4-мерным.

**6. Третий тип «тихого рассинхрона» за сессию.**

1. Две версии obsession-логики.
2. Два определения Discovery.
3. Сильные корреляции, не подтвердившиеся после детренда — 
   дважды (Sound ↔ Burnout, Sound ↔ Behavior Var).

Один класс ошибок: **выглядит как сигнал, не является сигналом**.

### → Ведёт к

Фаза 10: раз Behavior Variability не входит в компас — **почему**? 
Не потому что «не работает» (она работает, даёт 0.167). Причина — 
связь с Burnout (−0.757). Проверка — эмпирическая.

# Фаза 10. Behavior Variability не в компас

## Фаза 10. Behavior Variability не в компас

### Гипотеза

В фазе 8 получена Behavior Variability на 5 признаках (volume: `hours`, `tracks`; completion: `skip_rate`, `played_frac`; obsession: `obsession_share`). `discovery_rate` в неё не входит. В фазе 9 найдена связь с Burnout (−0.757 после детренда). Возможные причины не включать её в компас:

1. **Реальная связь.** Behavior и Burnout связаны содержательно.
2. **Дублирование Discovery.** Behavior Variability не должна повторять ось Discovery. Проверка: нет ли связи с Discovery у набора из фазы 8 и у набора с близким по смыслу `orphan_share`.

### Метод

1. Набор из фазы 8 (5 признаков) — берётся готовым, `var_beh`.
2. Дополнительный набор из 4 признаков: `hours`, `tracks`, `obsession_share`, `orphan_share` (`orphan_share` близок к Discovery по смыслу). Пороги p75 по каждому.
3. Кластерное голосование для 4 признаков:
   - volume: hours, tracks
   - obsession: obsession_share
   - orphan: orphan_share
4. High-месяц — если ≥2 кластера дали High.
5. Корреляция обоих рядов с осью Discovery.

In [13]:
# 10. Фаза 10. Behavior Variability не в компас
# ============================================================================
# Проверяем петлю: набор фазы 8 и набор с orphan_share»
# ============================================================================
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from scipy.stats import pearsonr

N_BOOT_PERM = 200
MIN_EVENTS = 5

# --- Discovery помесячно (уже считали в фазе 9) ---
# Пересчитаем для чистоты контекста
first_seen = (df_t.groupby('spotify_track_uri')['ts_naive'].min()
              .dt.to_period('M'))
df_10 = df_t.copy()
df_10['_first_month'] = df_10['spotify_track_uri'].map(first_seen)
df_10['_is_discovery'] = df_10['period'] == df_10['_first_month']
discovery_monthly = (df_10[df_10['valid_play']]
                     .groupby('period')['_is_discovery']
                     .mean()
                     .sort_index())
discovery_monthly.index = pd.PeriodIndex(discovery_monthly.index, freq='M')

print(f"Discovery помесячно: n={len(discovery_monthly)}, "
      f"median={discovery_monthly.median():.3f}")

# --- Behavior Variability на 6 признаках — берём из фазы 8 ---
var_beh_6 = var_beh.copy()

# --- Поведенческая база на 4 признаках ---
# orphan_share = 1 − доля my_music среди valid-событий месяца
valid_ev = df_10[df_10['valid_play']]
orphan_monthly = (valid_ev.groupby('period')['my_music']
                  .apply(lambda x: 1 - x.mean())
                  .sort_index())

mb4 = df_10.groupby('period').agg(
    hours=('ms_played', lambda x: x.sum() / 3_600_000),
    tracks=('spotify_track_uri', 'nunique'),
).sort_index()

# obsession_share — из sess_flag (построен в фазе 8)
mb4['obsession_share'] = (sess_flag.groupby('period')['is_obsession']
                          .mean()
                          .reindex(mb4.index).fillna(0.0))
mb4['orphan_share'] = orphan_monthly.reindex(mb4.index).fillna(0.0)

# Практические пороги p75
CAL4 = {}
for f in ['hours', 'tracks', 'obsession_share', 'orphan_share']:
    CAL4[f] = mb4[f].diff().abs().quantile(0.75)

print(f"\nПороги (p75) — чистый набор:")
for k, v in CAL4.items():
    print(f"   {k:18s}: {v:.4f}")

# --- Null per feature ---
def gate(obs, nulls, thr_prac, p_low=75, p_high=99):
    thr_low = np.percentile(nulls, p_low)
    thr_high = np.percentile(nulls, p_high)
    if obs > thr_high and obs >= thr_prac: return 'High'
    if obs > thr_low and obs >= thr_prac: return 'Medium'
    return 'Low'

def null_perm(a, b, fn, na, nb, n_boot, rng):
    pool = pd.concat([a, b], ignore_index=True)
    out = np.empty(n_boot)
    for k in range(n_boot):
        p = rng.permutation(len(pool))
        out[k] = abs(fn(pool.iloc[p[na:na + nb]]) -
                     fn(pool.iloc[p[:na]]))
    return out

agg_hours       = lambda e: e['ms_played'].sum() / 3_600_000
agg_tracks      = lambda e: e['spotify_track_uri'].nunique()
agg_obsession   = lambda s: s['is_obsession'].mean()
agg_orphan      = lambda e: 1 - e['my_music'].mean()

# --- Прогон ---
rng = np.random.default_rng(42)
months = sorted(df_10['period'].dropna().unique())
events = df_10[['account', 'ts_naive', 'period', 'ms_played',
                'spotify_track_uri', 'my_music']].copy()

per_month_4 = {}
for i in range(len(months) - 1):
    m, m1 = months[i], months[i + 1]
    e0 = events[events['period'] == m]
    e1 = events[events['period'] == m1]
    if len(e0) < MIN_EVENTS or len(e1) < MIN_EVENTS:
        continue

    row = {}
    for name, agg in [('hours', agg_hours),
                      ('tracks', agg_tracks),
                      ('orphan_share', agg_orphan)]:
        obs = abs(agg(e1) - agg(e0))
        nulls = null_perm(e0, e1, agg, len(e0), len(e1), N_BOOT_PERM, rng)
        row[name] = gate(obs, nulls, CAL4[name])

    s0 = sess_flag[sess_flag['period'] == m]
    s1 = sess_flag[sess_flag['period'] == m1]
    if len(s0) >= 3 and len(s1) >= 3:
        obs = abs(agg_obsession(s1) - agg_obsession(s0))
        nulls = null_perm(s0, s1, agg_obsession, len(s0), len(s1),
                          N_BOOT_PERM, rng)
        row['obsession_share'] = gate(obs, nulls, CAL4['obsession_share'])

    if row:
        per_month_4[str(m1)] = row

# --- Кластеры для 4 признаков ---
CLUSTERS_4 = {
    'volume':    ['hours', 'tracks'],
    'obsession': ['obsession_share'],
    'orphan':    ['orphan_share'],
}

events_beh_4 = pd.Series(0, index=pd.PeriodIndex(
    [str(m) for m in months[1:]], freq='M'))

for month_str, per_feat in per_month_4.items():
    cluster_high = 0
    for cname, feats in CLUSTERS_4.items():
        labs = [per_feat[f] for f in feats if f in per_feat]
        if 'High' in labs:
            cluster_high += 1
    if cluster_high >= 2:
        events_beh_4[pd.Period(month_str, 'M')] = 1

n_high_4 = int(events_beh_4.sum())
print(f"\nBehavior Variability (+orphan_share): High = {n_high_4} из "
      f"{len(events_beh_4)} ({events_beh_4.mean():.3f})")
print(f"Behavior Variability (фаза 8): High = {int(events_beh.sum())} "
      f"({events_beh.mean():.3f})")

# --- Variability для обоих ---
var_beh_4 = events_beh_4.rolling(12, min_periods=4).mean()

# --- Корреляции с Discovery ---
print("\n" + "=" * 70)
print("ПРОВЕРКА ПЕТЛИ: Behavior Var ↔ Discovery")
print("=" * 70)

def corr_pair(a, b, name):
    common = a.dropna().index.intersection(b.dropna().index)
    if len(common) < 10:
        return None
    aa = a.loc[common]; bb = b.loc[common]
    r, p = pearsonr(aa, bb)
    print(f"\n{name} (n={len(common)}): r = {r:+.3f}, p = {p:.4f}")
    return r, p

r6, p6 = corr_pair(var_beh_6, discovery_monthly,
                   "Behavior Var (фаза 8, 5 призн.) ↔ Discovery")
r4, p4 = corr_pair(var_beh_4, discovery_monthly,
                   "Behavior Var (+orphan_share, 4 призн.) ↔ Discovery")

print(f"\nВывод:")
print(f"   Набор из фазы 8: связь с Discovery {'значима' if p6 < 0.05 else 'не обнаружена'}")
print(f"   Набор с orphan_share: связь с Discovery {'значима' if p4 < 0.05 else 'не обнаружена'}")

# --- Визуализация: 4 линии ---
fig = go.Figure()

# Discovery ось
fig.add_trace(go.Scatter(
    x=discovery_monthly.dropna().index.to_timestamp(),
    y=discovery_monthly.dropna().values,
    mode='lines', name='Discovery (ось)',
    line=dict(width=2),
))

# Behavior Var 6
fig.add_trace(go.Scatter(
    x=var_beh_6.dropna().index.to_timestamp(),
    y=var_beh_6.dropna().values,
    mode='lines', name='Behavior Var (фаза 8, 5 призн.)',
    line=dict(width=2),
))

# Behavior Var 4
fig.add_trace(go.Scatter(
    x=var_beh_4.dropna().index.to_timestamp(),
    y=var_beh_4.dropna().values,
    mode='lines', name='Behavior Var (+orphan_share, 4 призн.)',
    line=dict(width=2),
))

fig.update_layout(
    template='plotly_white',
    height=400,
    title='Behavior Variability (фаза 8 vs +orphan_share) vs Discovery',
    xaxis_title='Месяц',
    yaxis_title='Значение',
    margin=dict(l=60, r=40, t=60, b=40),
)
fig.show()

Discovery помесячно: n=70, median=0.455

Пороги (p75) — чистый набор:
   hours             : 39.2928
   tracks            : 447.0000
   obsession_share   : 0.0805
   orphan_share      : 0.1984

Behavior Variability (+orphan_share): High = 5 из 69 (0.072)
Behavior Variability (фаза 8): High = 10 (0.145)

ПРОВЕРКА ПЕТЛИ: Behavior Var ↔ Discovery

Behavior Var (фаза 8, 5 призн.) ↔ Discovery (n=66): r = -0.094, p = 0.4525

Behavior Var (+orphan_share, 4 призн.) ↔ Discovery (n=66): r = +0.138, p = 0.2700

Вывод:
   Набор из фазы 8: связь с Discovery не обнаружена
   Набор с orphan_share: связь с Discovery не обнаружена


### Результат

**Проверка связи Behavior Var ↔ Discovery:**

| Набор | r | p |
|---|---|---|
| Фаза 8 (5 признаков) | −0.094 | 0.45 |
| С `orphan_share` (4 признака) | +0.138 | 0.27 |

**Ни один вариант не даёт связи с Discovery.**

**High-месяцев:**
- фаза 8: 10 из 69
- с orphan_share: 5 из 69

Практические пороги (p75) для 4 признаков:

| Признак | Порог |
|---|---|
| hours | 39.29 |
| tracks | 447.0 |
| obsession_share | 0.081 |
| orphan_share | 0.198 |

### 🔄 Поворот

**1. Дублирование Discovery не обнаружено.**

Behavior Variability из фазы 8 не связана с осью Discovery (r = −0.094, p = 0.45), и набор с добавленным близким по смыслу `orphan_share` тоже (r = +0.138, p = 0.27). `discovery_rate` в неё не входит.

**2. Behavior Variability не входит в компас — по эмпирической причине.**

Фаза 9 показала: **Behavior Var ↔ Burnout = −0.757** после детренда. 
Это **реальная связь**. Если добавить Behavior как 5-ю ось — Burnout 
и Behavior начнут дублировать друг друга. Компас перестанет быть 
ортогональным.

**3. Компас 4-осевой — эмпирически обоснован.**

Все 6 пар **внутри компаса** (Discovery, Retention, Burnout, 
Sound Var) после детренда не значимы. Behavior Variability связана 
с Burnout — поэтому **не входит**.

**4. Behavior Variability остаётся исследовательской метрикой.** Метод валиден (5 High-месяцев на наборе с orphan_share, 10 на наборе фазы 8), но в компас не входит и в дашборд не экспортируется.

### → Ведёт к

Фаза 11: метод работает на трёх уровнях (Sound, Behavior, 
архитектурно отвергнутый Behavior для компаса). Пора **обобщить** 
в переиспользуемый инструмент. Не только для этого пользователя, 
а для любых данных вида «период → наблюдения → агрегат».

### Проверка устойчивости: leave-one-out

Для каждого из 10 High-месяцев Behavior Variability пересчитана без него, затем снова посчитана корреляция с Burnout после детренда. Базовое значение r = −0.757 воспроизводится; в 10 вариантах r от −0.614 до −0.842 (медиана −0.703), знак отрицательный во всех. Связь не опирается на отдельные месяцы.

In [14]:
# Leave-one-out по High-месяцам: устойчив ли r(Behavior Var, Burnout) после детренда
# ============================================================================
# Для каждого из High-месяцев (events_beh == 1) обнуляем его, пересчитываем
# rolling(12, min_periods=4) (как в ф.8) и считаем detrended r с Burnout.
# Базовая строка должна воспроизвести r = -0.757 из ф.9.
# ============================================================================
import numpy as np
import pandas as pd
from scipy.stats import pearsonr

def _detr_corr(series_beh, series_burn):
    common = series_beh.dropna().index.intersection(series_burn.dropna().index)
    a, b = series_beh.loc[common], series_burn.loc[common]
    r, p = pearsonr(detrend(a), detrend(b))
    return r, p, len(common)

base_var = events_beh.rolling(12, min_periods=4).mean()
r0, p0, n0 = _detr_corr(base_var, burnout_monthly)
print(f"Базовая: r = {r0:+.3f}, p = {p0:.4f}, n = {n0}  (ожидание: -0.757, n=58)\n")

high_months = list(events_beh.index[events_beh == 1])
rows = []
for m in high_months:
    ev = events_beh.copy()
    ev.loc[m] = 0
    v = ev.rolling(12, min_periods=4).mean()
    r, p, n = _detr_corr(v, burnout_monthly)
    rows.append({'убран месяц': str(m), 'r_detr': round(r, 3), 'p': round(p, 4), 'n': n})

loo = pd.DataFrame(rows)
print(loo.to_string(index=False))
print(f"\nr по 10 вариантам: min {loo['r_detr'].min():+.3f}, "
      f"max {loo['r_detr'].max():+.3f}, медиана {loo['r_detr'].median():+.3f}")
print(f"Знак остаётся отрицательным во всех: {(loo['r_detr'] < 0).all()}")

Базовая: r = -0.757, p = 0.0000, n = 58  (ожидание: -0.757, n=58)

убран месяц  r_detr   p  n
    2021-03  -0.714 0.0 58
    2021-04  -0.688 0.0 58
    2023-09  -0.755 0.0 58
    2024-06  -0.706 0.0 58
    2024-11  -0.773 0.0 58
    2025-02  -0.842 0.0 58
    2025-08  -0.666 0.0 58
    2025-10  -0.643 0.0 58
    2025-12  -0.614 0.0 58
    2026-06  -0.700 0.0 58

r по 10 вариантам: min -0.842, max -0.614, медиана -0.703
Знак остаётся отрицательным во всех: True


# Фаза 11. Toolkit — обобщение

## Фаза 11. Toolkit — обобщение

### Гипотеза

Метод Variability работает: Sound на 6 признаках, Behavior на 
кластерном голосовании. Один и тот же принцип — **наблюдаемое vs 
permutation noise floor**. Если он работает на разных структурах, 
можно **обобщить** в переиспользуемый модуль.

Плюс — вынести ключевой принцип: **«единица обмена»** — что можно 
перемешивать без потери смысла. Она зависит от структуры данных, 
а не от метода:
- Event-level признаки → перемешиваются **события**.
- Session-level → **сессии**.
- Категориальные высокой кардинальности → permutation **не работает** 
  (проверка в приложении C).

Toolkit делает это **явным параметром**, а не скрытым предположением.

### Метод

1. Вызов `VariabilityAnalyzer` из `src/detectors/variability_toolkit.py`.
2. Настройка на те же параметры, что продакшн-пайплайн фазы 7: 
   `p_low=75`, `p_high=99`, `min_features=2`, `min_periods=12`.
3. Сравнение результата с продакшн-пайплайном.
4. Проверка устойчивости на 5 seed.

In [15]:
# 11. Фаза 11. Toolkit — обобщение
# ============================================================================
# VariabilityAnalyzer из src/detectors/variability_toolkit.py.
# Toolkit — Sound-метод. Кластерное голосование Behavior (фаза 8) —
# надстройка поверх, отдельная логика в run_variability_pipeline.
# Проверяем: (1) совпадение с продакшном, (2) стохастика по seed.
# ============================================================================
from detectors.variability_toolkit import VariabilityAnalyzer

print("=" * 70)
print("1. SOUND — сверка с продакшн-пайплайном фазы 7")
print("=" * 70)

df_valid = df_t[df_t['valid_play']].copy()
monthly_valid = build_monthly_sound(df_valid)
p75_sound = monthly_valid[SOUND_FEATURES].diff().abs().quantile(0.75).to_dict()

events_sound = df_valid[['period'] + SOUND_FEATURES].dropna().copy()
events_sound['period_str'] = events_sound['period'].astype(str)

def run_toolkit_sound(seed):
    az = VariabilityAnalyzer(
        period_col='period_str',
        value_cols=SOUND_FEATURES,
        agg='median',
        n_boot=200, p_low=75, p_high=99,
        min_obs_per_period=5,
        practical_thresholds=p75_sound,
        min_practical_features=2,
        random_state=seed,
    )
    az.fit(events_sound)
    return az.variability_score(window=12, n_boot=500, min_periods=12)

score_sound = run_toolkit_sound(seed=42)
print(f"Toolkit Sound lifetime:    {score_sound['lifetime_score']:.3f}")
print(f"Ручной пайплайн (фаза 7):  {R['lifetime_score']:.3f}")
match = abs(score_sound['lifetime_score'] - R['lifetime_score']) < 0.01
print(f"Совпадение: {'✅' if match else '⚠️'}")

# --- Стохастика: 5 seed ---
print("\n" + "=" * 70)
print("2. Стохастика — 5 seed")
print("=" * 70)

seeds = [42, 1, 7, 13, 100]
seed_results = []
for s in seeds:
    sc = run_toolkit_sound(seed=s)
    seed_results.append({
        'seed': s,
        'lifetime': sc['lifetime_score'],
        'null_high': sc['null_high'],
    })
    print(f"   seed={s:>4}: lifetime = {sc['lifetime_score']:.3f}  "
          f"(null_high = {sc['null_high']:.3f})")

df_seeds = pd.DataFrame(seed_results)
print(f"\nРазброс lifetime: "
      f"{df_seeds['lifetime'].min():.3f} – {df_seeds['lifetime'].max():.3f}")
print(f"null_high во всех seed: {df_seeds['null_high'].min():.3f} — "
      f"{df_seeds['null_high'].max():.3f}")

# --- API toolkit ---
print("\n" + "=" * 70)
print("3. API toolkit")
print("=" * 70)
print("""
from detectors.variability_toolkit import VariabilityAnalyzer

az = VariabilityAnalyzer(
    period_col='month',
    value_cols=['val', 'energy', ...],
    agg='median',
    n_boot=200, p_low=75, p_high=99,
    min_obs_per_period=5,
    practical_thresholds=None,            # auto p50
    min_practical_features=2,
    random_state=42,
)
az.fit(df)
score = az.variability_score(window=12, n_boot=500, min_periods=12)
# → lifetime_score, current_score, null_low, null_high, epochs
""")

1. SOUND — сверка с продакшн-пайплайном фазы 7
Toolkit Sound lifetime:    0.417
Ручной пайплайн (фаза 7):  0.417
Совпадение: ✅

2. Стохастика — 5 seed
   seed=  42: lifetime = 0.417  (null_high = 0.417)
   seed=   1: lifetime = 0.417  (null_high = 0.417)
   seed=   7: lifetime = 0.417  (null_high = 0.417)
   seed=  13: lifetime = 0.417  (null_high = 0.458)
   seed= 100: lifetime = 0.417  (null_high = 0.417)

Разброс lifetime: 0.417 – 0.417
null_high во всех seed: 0.417 — 0.458

3. API toolkit

from detectors.variability_toolkit import VariabilityAnalyzer

az = VariabilityAnalyzer(
    period_col='month',
    value_cols=['val', 'energy', ...],
    agg='median',
    n_boot=200, p_low=75, p_high=99,
    min_obs_per_period=5,
    practical_thresholds=None,            # auto p50
    min_practical_features=2,
    random_state=42,
)
az.fit(df)
score = az.variability_score(window=12, n_boot=500, min_periods=12)
# → lifetime_score, current_score, null_low, null_high, epochs



### Результат

**Toolkit Sound = продакшн-пайплайн = 0.417.** ✅

**Устойчивость: lifetime стабилен во всех 5 seed.**

| seed | lifetime | null_high |
|---|---|---|
| 42 | 0.417 | 0.417 |
| 1 | 0.417 | 0.417 |
| 7 | 0.417 | 0.417 |
| 13 | 0.417 | **0.458** |
| 100 | 0.417 | 0.417 |

Разброс lifetime: **0.417 – 0.417** (ноль). Разброс `null_high`: 
0.417 – 0.458 (0.04).

**API toolkit:** три параметра + один принцип:
- `value_cols` — признаки
- `agg` — median / mean
- `practical_thresholds` — калиброванные p75
- `min_practical_features` — сколько признаков должны сдвинуться

### Примечание: min_periods

Без явного `min_periods` toolkit берёт `max(3, window // 3) = 4`: первые 8 точек считаются по неполным окнам, медиана смещена вверх. Проверено: `min_periods=None` даёт lifetime = 0.500, `min_periods=12` даёт 0.417. Продакшн передаёт 12 явно. Значение 0.423 из раннего прогона в этом ноутбуке не воспроизводится.

### 🔄 Поворот

**1. «Единица обмена» — ключевой принцип.**

Permutation работает только если **перемешиваемая единица сохраняет 
смысл**. Четыре случая:

| Данные | Единица обмена | Работает? |
|---|---|---|
| Sound (valence, energy, …) | событие | ✅ |
| Behavior (skip_rate, played_frac) | событие | ✅ |
| Behavior (obsession_share) | **сессия** | ✅ |
| Cosine (состав артистов) | **нет** | ❌ |

**Cosine — провал permutation-null.** Перемешивание событий двух 
месяцев даёт почти одинаковые распределения (закон больших чисел) — 
состав артистов всегда разный. «Нет изменения» — неправдоподобная 
нулевая гипотеза для категориальных с высокой кардинальностью. 
Это не проблема метода, а **свойство данных**.

Toolkit делает выбор единицы обмена **явным параметром**, а не 
скрытым.

**2. Null-граница плавает, продуктовая метка — нет.**

`null_high` зависит от seed: 0.417 – 0.458. Но она **не используется 
в продакшне**. В `indices_display.json` — фиксированные пороги 
**0.3 / 0.5**, как у трёх других осей компаса.

Это архитектурное решение:
- **Null-based** уровни — для отладки и анализа.
- **Фиксированные пороги** — для продуктовой метки. Стабильно, 
  воспроизводимо, не зависит от seed.

**3. Behavior — не через toolkit напрямую.**

Кластерное голосование (volume / completion / obsession) — 
**надстройка над toolkit**, не его прямое применение. Живёт в 
`run_variability_pipeline`. Toolkit — Sound-метод: единый гейт 
по всем признакам.

### → Ведёт к

Фаза 12: метод работает, toolkit обобщён, устойчивость проверена. 
Остаётся последний вопрос — **на какой базе считать**. До сих пор 
PELT-A/B шли на **ALL**, Variability — на **VALID**. Проверка 
устойчивости результата к выбору базы.

# Фаза 12. ALL vs VALID

## Фаза 12. ALL vs VALID

### Гипотеза

До сих пор слои работали на **разных базах**:

- **PELT-A/B** — ALL события. Причина: `skip_rate` и `played_frac` 
  тавтологически не определены на valid (skip по определению = 0).
- **Variability, компас** — VALID события. Причина: скипы — 
  не осознанное слушание, маскируют сигнал.

Если база **не влияет** на результат — можно унифицировать. Если 
влияет — зафиксировать раздельно и документировать, почему.

### Метод

Все слои прогоняются на двух базах:

- **ALL** — `df_t` (после trim_cold_start).
- **VALID** — `df_t[df_t['valid_play']]`.

Сравнение: точки PELT, эпизоды Storm, High-месяцы Variability.

In [16]:
# 12. Фаза 12. ALL vs VALID
# ============================================================================
# Прогоняем все слои на двух базах. Сравниваем: точки PELT,
# эпохи Variability, эпизоды Storm. Цель — понять, где база
# влияет на результат, а где нет.
# ============================================================================
import plotly.graph_objects as go
from detectors.pelt_sound import detect as pelt_sound_detect
from detectors.cosine_identity import detect as cosine_detect
from detectors.storm import detect as storm_detect

STABILITY_THRESHOLD = 0.10

# --- Две базы ---
df_all = df_t.copy()
df_valid_b = df_t[df_t['valid_play']].copy()

print(f"ALL:   {len(df_all):,} событий, "
      f"{df_all['period'].nunique()} месяцев")
print(f"VALID: {len(df_valid_b):,} событий, "
      f"{df_valid_b['period'].nunique()} месяцев "
      f"({len(df_valid_b)/len(df_all)*100:.1f}%)")


# ==============================================================================
# PELT-A (звук): сравнение
# ==============================================================================
print("\n" + "=" * 70)
print("PELT-A (звук): ALL vs VALID")
print("=" * 70)

ms_all = build_monthly_sound(df_all)
ms_valid = build_monthly_sound(df_valid_b)

res_a_all = pelt_sound_detect(ms_all)
res_a_valid = pelt_sound_detect(ms_valid)

points_a_all = [p for p in res_a_all['points']
                if pd.Series(res_a_all['stability']).get(p, 0) >= STABILITY_THRESHOLD]
points_a_valid = [p for p in res_a_valid['points']
                  if pd.Series(res_a_valid['stability']).get(p, 0) >= STABILITY_THRESHOLD]

print(f"ALL:   σ²={res_a_all['sigma2']:.3f}, точки: {points_a_all}")
print(f"VALID: σ²={res_a_valid['sigma2']:.3f}, точки: {points_a_valid}")

common_a = set(points_a_all) & set(points_a_valid)
print(f"\nПересечение: {len(common_a)} — {sorted(common_a)}")
print(f"Только ALL:   {sorted(set(points_a_all) - set(points_a_valid))}")
print(f"Только VALID: {sorted(set(points_a_valid) - set(points_a_all))}")


# ==============================================================================
# COSINE (артисты): сравнение
# ==============================================================================
print("\n" + "=" * 70)
print("Cosine (артисты): ALL vs VALID")
print("=" * 70)

res_cos_all = cosine_detect(df_all, id_col='artist_name')
res_cos_valid = cosine_detect(df_valid_b, id_col='artist_name')

points_cos_all = list(res_cos_all['points'])
points_cos_valid = list(res_cos_valid['points'])

print(f"ALL:   точки: {points_cos_all}")
print(f"VALID: точки: {points_cos_valid}")

common_cos = set(points_cos_all) & set(points_cos_valid)
print(f"\nПересечение: {len(common_cos)} — {sorted(common_cos)}")
print(f"Только ALL:   {sorted(set(points_cos_all) - set(points_cos_valid))}")
print(f"Только VALID: {sorted(set(points_cos_valid) - set(points_cos_all))}")


# ==============================================================================
# STORM: сравнение
# ==============================================================================
print("\n" + "=" * 70)
print("Storm: ALL vs VALID")
print("=" * 70)

res_storm_all = storm_detect(ms_all)
res_storm_valid = storm_detect(ms_valid)

ep_all = [f"{ep[0]}..{ep[-1]}" for ep in res_storm_all['episodes']]
ep_valid = [f"{ep[0]}..{ep[-1]}" for ep in res_storm_valid['episodes']]

print(f"ALL:   порог={res_storm_all['threshold']:.4f}, эпизоды: {ep_all}")
print(f"VALID: порог={res_storm_valid['threshold']:.4f}, эпизоды: {ep_valid}")


# ==============================================================================
# VARIABILITY: ALL vs VALID
# ==============================================================================
print("\n" + "=" * 70)
print("Variability (Sound): ALL vs VALID")
print("=" * 70)

from detectors.variability import run_variability_pipeline

R_all = run_variability_pipeline(
    df_all,
    monthly_sound=ms_all,
    window=12,
    seed=42,
)
R_valid_b = run_variability_pipeline(
    df_valid_b,
    monthly_sound=ms_valid,
    window=12,
    seed=42,
)

print(f"ALL:   lifetime = {R_all['lifetime_score']:.3f}, "
      f"current = {R_all['current_score']:.3f}")
print(f"VALID: lifetime = {R_valid_b['lifetime_score']:.3f}, "
      f"current = {R_valid_b['current_score']:.3f}")

# Уровни по фиксированным порогам 0.3/0.5
def level(s):
    if s < 0.3: return 'Low'
    if s <= 0.5: return 'Medium'
    return 'High'

print(f"\nLevel (0.3/0.5):")
print(f"   ALL:   {level(R_all['lifetime_score'])}")
print(f"   VALID: {level(R_valid_b['lifetime_score'])}")

# High-месяцы
high_all = set(R_all['transitions'][R_all['transitions']['label'] == 'High'].index)
high_valid = set(R_valid_b['transitions'][R_valid_b['transitions']['label'] == 'High'].index)

print(f"\nHigh-месяцев:")
print(f"   ALL:   {len(high_all)}")
print(f"   VALID: {len(high_valid)}")
print(f"   Совпадают: {len(high_all & high_valid)}")
print(f"   Только ALL:   {len(high_all - high_valid)} "
      f"{sorted(list(high_all - high_valid))[:5]}")
print(f"   Только VALID: {len(high_valid - high_all)} "
      f"{sorted(list(high_valid - high_all))[:5]}")


# ==============================================================================
# СВОДКА
# ==============================================================================
print("\n" + "=" * 70)
print("СВОДКА: ALL vs VALID")
print("=" * 70)

summary = pd.DataFrame([
    {
        'layer': 'PELT-A (звук)',
        'ALL': len(points_a_all),
        'VALID': len(points_a_valid),
        'common': len(common_a),
    },
    {
        'layer': 'Cosine (артисты)',
        'ALL': len(points_cos_all),
        'VALID': len(points_cos_valid),
        'common': len(common_cos),
    },
    {
        'layer': 'Storm (эпизоды)',
        'ALL': len(res_storm_all['episodes']),
        'VALID': len(res_storm_valid['episodes']),
        'common': '—',
    },
    {
        'layer': 'Variability High-месяцев',
        'ALL': len(high_all),
        'VALID': len(high_valid),
        'common': len(high_all & high_valid),
    },
    {
        'layer': 'Variability lifetime',
        'ALL': f"{R_all['lifetime_score']:.3f}",
        'VALID': f"{R_valid_b['lifetime_score']:.3f}",
        'common': '—',
    },
])
print(summary.to_string(index=False))


# ==============================================================================
# ВИЗУАЛИЗАЦИЯ
# ==============================================================================
fig = go.Figure()

# PELT-A ALL
x = [pd.Period(p, 'M').to_timestamp() for p in points_a_all]
fig.add_trace(go.Scatter(
    x=x, y=[2] * len(x), mode='markers',
    marker=dict(size=12, symbol='line-ns-open', line=dict(width=3)),
    name=f'PELT-A ALL ({len(points_a_all)})',
))

# PELT-A VALID
x = [pd.Period(p, 'M').to_timestamp() for p in points_a_valid]
fig.add_trace(go.Scatter(
    x=x, y=[1.8] * len(x), mode='markers',
    marker=dict(size=12, symbol='line-ns-open', line=dict(width=3)),
    name=f'PELT-A VALID ({len(points_a_valid)})',
))

# Cosine ALL
x = [pd.Period(p, 'M').to_timestamp() for p in points_cos_all]
fig.add_trace(go.Scatter(
    x=x, y=[1] * len(x), mode='markers',
    marker=dict(size=12, symbol='line-ns-open', line=dict(width=3)),
    name=f'Cosine ALL ({len(points_cos_all)})',
))

# Cosine VALID
x = [pd.Period(p, 'M').to_timestamp() for p in points_cos_valid]
fig.add_trace(go.Scatter(
    x=x, y=[0.8] * len(x), mode='markers',
    marker=dict(size=12, symbol='line-ns-open', line=dict(width=3)),
    name=f'Cosine VALID ({len(points_cos_valid)})',
))

fig.update_layout(
    template='plotly_white',
    height=350,
    title='ALL vs VALID: точки по слоям',
    xaxis_title='Месяц',
    yaxis=dict(tickmode='array',
               tickvals=[0.8, 1, 1.8, 2],
               ticktext=['Cosine VALID', 'Cosine ALL',
                         'PELT-A VALID', 'PELT-A ALL'],
               range=[0.5, 2.3]),
    margin=dict(l=100, r=40, t=60, b=40),
)
fig.show()

ALL:   79,531 событий, 70 месяцев
VALID: 45,706 событий, 70 месяцев (57.5%)

PELT-A (звук): ALL vs VALID
ALL:   σ²=0.420, точки: ['2023-09', '2024-02', '2024-07']
VALID: σ²=0.521, точки: ['2023-09', '2024-02', '2024-07']

Пересечение: 3 — ['2023-09', '2024-02', '2024-07']
Только ALL:   []
Только VALID: []

Cosine (артисты): ALL vs VALID
ALL:   точки: ['2021-02', '2022-05', '2024-02']
VALID: точки: ['2021-02']

Пересечение: 1 — ['2021-02']
Только ALL:   ['2022-05', '2024-02']
Только VALID: []

Storm: ALL vs VALID
ALL:   порог=0.0996, эпизоды: ['2021-01..2021-06', '2023-10..2024-05']
VALID: порог=0.1037, эпизоды: ['2021-01..2021-06', '2023-10..2024-04', '2024-11..2025-01']

Variability (Sound): ALL vs VALID
ALL:   lifetime = 0.417, current = 0.000
VALID: lifetime = 0.417, current = 0.083

Level (0.3/0.5):
   ALL:   Medium
   VALID: Medium

High-месяцев:
   ALL:   27
   VALID: 27
   Совпадают: 23
   Только ALL:   4 ['2022-02', '2024-03', '2024-09', '2025-02']
   Только VALID: 4 ['2020-11'

### Результат

| Слой | ALL | VALID | Совпадение |
|---|---|---|---|
| PELT-A (звук) | 3 | 3 | **3** ✅ |
| Cosine (артисты) | 3 | 1 | 1 |
| Storm (эпизоды) | 2 | 3 | — |
| Variability High-месяцев | 27 | 27 | 23 |
| Variability lifetime | 0.417 | 0.417 | — |
| Variability current | 0.000 | 0.083 | — |

**PELT-A — устойчив к базе.** Те же 3 точки. σ² растёт на VALID 
(0.42 → 0.52) — меньше данных, но точки идентичны. Звуковые 
признаки трека объективны: даже если skip'нул, `valence`/`energy` 
отражают сам трек.

**Cosine — теряет 2 из 3 точек на VALID.** Остаётся только 
`2021-02`. `2022-05` и `2024-02` исчезают. На VALID матрица 
«месяц × артист» разряжается — косинусные расстояния между 
соседними месяцами падают ниже порога.

**Storm — даёт новую зону на VALID:** `2024-11..2025-01`. Плюс 
сжимает существующую. Rolling std чувствителен к базе.

**Variability lifetime идентичен (0.417 Medium).** Но:
- `current` различается: ALL = 0.000, VALID = 0.083.
- High-месяцев 27 и 27, но совпадают только 23. Четыре месяца 
  «переехали» с одной базы на другую.

### 🔄 Поворот

**1. Семантика победила единообразие.**

Одна база для всего не подходит. Причины содержательные:

**VALID — где сигнал про «осознанное слушание»:**
- **Variability**: High-месяцы — про изменение вкуса. Скипы — не «изменение», а «случайное касание». VALID.
- **Portrait**: Discovery, Retention, Burnout — про сознательный выбор. VALID.

**ALL — где скипы часть сигнала или база не влияет:**
- **PELT-A**: звук трека не зависит от skip, результат от базы не меняется (те же 3 точки), ALL даёт больше данных (σ² 0.420 против 0.521).
- **PELT-B**: `skip_rate` и `played_frac` тавтологически не определены на VALID (skip = 0, `played_frac` ≥ 0.5). Детектор сломается.
- **Cosine**: на VALID теряется 2 из 3 точек. Матрица слишком разряжена.
- **Storm**: rolling std чище на ALL. Новый эпизод на VALID — может быть артефактом малого N (45k vs 79k).
- **Obsessions**: strong и base — про интенсивность, включая скипы. ALL.

**2. Variability lifetime совпадает, но High-месяцы — нет.**

0.417 на обеих базах. Но 4 месяца переехали:
- Только ALL: `2022-02`, `2024-03`, `2024-09`, `2025-02`
- Только VALID: `2020-11`, `2024-12`, `2025-01`, `2025-11`

Медиана сходится, состав High — нет. Значит эпохи и текущее 
состояние могут отличаться. `current`: ALL = 0.000, 
VALID = 0.083.

**3. Канонические базы.**

| Потребитель | База | Результат |
|---|---|---|
| PELT-A | all + trimmed | 3 точки |
| PELT-B | all + trimmed | 5 точек |
| Cosine | all + trimmed | 3 точки |
| Storm | all + trimmed | 2 эпизода |
| Variability | valid + trimmed | 0.417 Medium |
| Portrait (3 оси) | valid | Discovery / Retention / Burnout |
| Obsessions | all + trimmed | strong + base |

### → Ведёт к

Финальная система зафиксирована. Четыре слоя точек + ось Variability.
`2024-02` — трижды усиленная точка: все три слоя точек сходятся 
в этом месяце. Компас функционально 4-мерный. Поворот завершён.

# Финальная система

## Финальная система

## Что идёт в продакшн

**Три независимых слоя точек и слой зон:**

| Слой | База | Точки |
|---|---|---|
| PELT-A (звук) | all + trimmed | `2023-09`, `2024-02`, `2024-07` |
| PELT-B (поведение) | all + trimmed | `2021-08`, `2024-02`, `2024-12`, `2025-10`, `2026-03` |
| Cosine (артисты) | all + trimmed | `2021-02`, `2022-05`, `2024-02` |
| Storm (зоны нестабильности) | all + trimmed | `2021-01..2021-06`, `2023-10..2024-05` |

**Фоновый слой Variability:**
- Sound Variability: **0.417 Medium** (Lifetime), **0.083 Low** (Current).
- Три эпохи: `2021-10..2022-06`, `2024-01..2024-04`, `2024-06..2025-01`.

**Компас (4 оси):**
- Discovery, Retention, Burnout, Sound Variability.

Пары либо не связаны, либо связь не установлена при n≈58 (фаза 9). Оси построены на разных данных и измеряют разные вещи; эмпирически «ортогональность» здесь означает «сильной парной связи не обнаружено», а не математическую независимость.

**`2024-02` — трижды усиленная точка.** Все три слоя точек сходятся 
в одном месяце. Единственная дата, где одновременно сменились 
звук, поведение **и** артисты.

**Пороги в дашборде:** фиксированные 0.3 / 0.5, не null-based. 
Зафиксировано в `indices_display.json`.

**Модули в `src/detectors/`:**
- `pelt_sound.py`, `pelt_behavior.py`
- `cosine_identity.py`, `storm.py`
- `variability.py`, `variability_toolkit.py`

---

## Канонические базы

| Потребитель | База | Результат |
|---|---|---|
| PELT-A | all + trimmed | 3 точки |
| PELT-B | all + trimmed | 5 точек |
| Cosine | all + trimmed | 3 точки |
| Storm | all + trimmed | 2 эпизода |
| Variability | valid + trimmed | 0.417 Medium |
| Portrait (3 оси) | valid | Discovery / Retention / Burnout |
| Obsessions | all + trimmed | strong + base |

**Почему раздельно.** VALID отвечает на вопрос «что выбрано слушать 
сознательно». ALL — «что произошло в плеере в целом». Разные вопросы, 
разные базы.

---

## Отброшено

| Метод | Причина |
|---|---|
| Смешанный PELT (фаза 1) | Признаки разной природы глушат друг друга |
| `pen=10` | Не выводится из данных |
| SNR как критерий (прил. A) | Permutation-тест: прирост не отличается от шума |
| Периодизация (фаза 5) | 0 из 6 плато стабильны |
| Self-percentile (фаза 6) | Структурно даёт 33/33/33 |
| Cosine на жанрах (фаза 3) | Высокая кардинальность — 15 пиков-шум |
| Permutation-null для Cosine (прил. C) | Не работает для категориальных |
| Cut-scan как точки (прил. B) | Найдены регионы, не точки |
| `desktop_share` в PELT-B (фаза 4) | Один структурный скачок Android→Windows |
| `excess_ratio` как метрика (фазы 6–7) | Завышен большими N |

---

## Открытые вопросы

**1. `2022-11` в PELT-A.**
Отсеяна в bootstrap (0.050). Cut-scan показывает зону 
`2021-12..2023-03` — возможно, разладка размазана на 15 месяцев. 
Содержательно не проверялось.

**2. `2024-02` в PELT-B.**
Устойчива (0.135), но слабее других точек PELT-B (0.105–0.365). 
Поведенческая разладка или отголосок звуковой точки? Оба слоя 
дают одну дату.

**3. Behavior Var ↔ Burnout = −0.757 после детренда.**
Устойчива к удалению отдельных High-месяцев (leave-one-out: от −0.614 до −0.842). Не учтены перекрывающиеся окна (эффективный n меньше 58) и множественность 10 пар; поправка не применялась. Behavior Variability разрежен: 10 High-месяцев из 69.

**4. Пороги 0.3 / 0.5.**
Калиброваны эмпирически. Работают для этой истории. Для других 
могут не работать.

**5. Порог входа 24 перехода.**
Variability не считается при истории короче 24 переходов (`MIN_TRANSITIONS_FOR_VARIABILITY`); дашборд показывает «Not enough history».

**6. Пороги**
Статистический порог p99 выбран из-за ~70 одновременных проверок.
Чувствительность проверена (p90 / p95 / p99): lifetime 0.417 и уровень
Medium не меняются, число High-месяцев 29 / 29 / 27. Статистический
порог результат не определяет (при p99 он пропускает 49 переходов из 69
и почти не фильтрует).

Практический порог p75 определяет итог. При смене квантиля от p50 до
p95 lifetime меняется от 0.667 (High) до 0.083 (Low); уровень
принимает все три значения (High при p50-p70, Medium при p75-p80,
Low при p85-p95). Проверенная альтернатива — абсолютный порог
`PRACTICAL_THRESHOLDS × K` (база 0.02 для признаков в [0, 1], 2 BPM
для tempo): при K=5 lifetime и уровень совпадают с p75 (0.417, Medium),
High-месяцы пересекаются на 22 из 27 (81%). K=5 подобран по критерию
«то же, что p75», поэтому это документированная альтернатива, а не
независимое обоснование: чувствительность к K сохраняется (0.750 при
K=1, 0.167 при K=7.5).

Синтетический тест на 15 пользователях с известной изменчивостью
(5 stable, 5 moderate, 5 variable) даёт одинаковые медианы для p75 и
K=5: 0.000 / 0.000 / 0.167. Оба порога разделяют stable и variable
(0.000 против 0.167) и не отличают moderate от stable.

Перцептивная калибровка возможна только для tempo (JND темпа около
3–6%, Perception & Psychophysics, doi 10.3758/BF03205262); для
остальных признаков Spotify психофизических порогов нет.

В продакшне остаётся p75. Калибровка на N ≥ 50 не сделана: порог
откалиброван на этих данных и для другого профиля может потребовать
перекалибровки.

---

## За пределами этого ноутбука

- **Содержательная интерпретация точек** — почему именно эти 
  месяцы. Задача анализа поверх детекторов.
- **Перенос на других пользователей** — нужна отдельная 
  калибровка порогов.
- **Связь точек с внешними событиями** — данных нет, и это 
  не задача детектора.

---

## Что дальше

1. **`dashboard_data_prep.ipynb`** — собирает `time_changepoints.parquet` 
   (11 строк) и `variability.parquet` (69 строк) с канонических баз.
2. **Дашборд** — 3 слоя точек + штормовые эпизоды + Variability 
   в Time Personal, компас — в Behavior Personal.
3. **`PELT_JOURNEY.md`** — текстовая версия этого ноутбука, 12 фаз 
   со ссылками на конкретные ячейки.
4. **`METHODOLOGY_VARIABILITY.md`** — переносимый метод для любых 
   данных «период → наблюдения → агрегат». Возможно, планируется 
   отдельная публикация на GitHub (вынос `variability_toolkit.py` 
   в самостоятельный репозиторий).

# Куда это можно вести дальше

## Куда это можно вести дальше

Исходная гипотеза — музыка отражает человека, изменения в музыке
отражают изменения в человеке — остаётся мотивацией проекта. В этих
данных проверить её нечем: нет внешних меток событий.

## Как это можно развить

Если сдвиг находится не только в музыке, а одновременно в нескольких
независимых доменах личных данных (календарь, траты, физическая
активность) — это существенно более сильное подтверждение, чем сдвиг
в одном домене.

**Условие:** домены должны быть причинно независимы друг от друга
(не просто разные метрики одной платформы). Иначе конвергенция
окажется одним и тем же внешним толчком, посчитанным трижды,
а не тремя независимыми подтверждениями.

## Что дальше

- **Калибровка порога Variability на выборке пользователей (N ≥ 50):**
  на одном профиле порог проверяется только на согласованность с
  собственным выбором.
- **Независимые домены:** сдвиг, найденный одновременно в музыке и в
  данных другой природы (календарь, активность), был бы сильнее
  подтверждением, чем сдвиг в одном домене.
- **Ground truth:** пока нет внешней метки событий, метрика проверяется
  только на согласованность между слоями, а не на соответствие
  состоянию человека. Согласованность с событиями — не валидация.
- **Блочный bootstrap для корреляций фазы 9** (перекрывающиеся окна).

Индустриальные системы обычно обучаются на данных многих людей; здесь
метод строится и калибруется на N=1, поэтому пороги проверяются только
на согласованность.

---


# Приложения

### Приложения

Основной нарратив — 12 фаз — закончился. Всё, что было **поворотом**, 
**гипотезой с исходом** или **решением с последствием**, вошло в фазы.

Осталось **три блока проверок**, которые не двинули нарратив. 
Не потому что слабые — потому что **подтверждали** уже известное 
или **проверяли границы** метода за пределами основной задачи.

**Зачем они здесь:**

1. **Обосновать решения в основных фазах.**  В фазе 4 SNR не используется.
   как критерий сигнала — но **почему**? Без приложения A это 
   выглядит произволом.
2. **Показать границы метода.** В фазе 11 сформулирован принцип 
   «единица обмена» — что можно перемешивать без потери смысла. 
   Приложение C показывает, где этот принцип **нарушается** 
   и permutation ломается.
3. **Обосновать фоновый слой Storm**: заливка штормов в дашборде получена из приложения B. Без него в дашборде они «из ниоткуда».

**Что в приложениях:**

| Приложение | Тема | Итог |
|---|---|---|
| **A** | SNR — вклад признаков | До сглаживания 5 из 7 признаков ниже 1.5; SNR не критерий (permutation опроверг) |
| **B** | Cut-scan + Storm — регионы | Не точки, а зоны. Точки PELT/cosine **не** концентрируются в штормах (`p = 0.66`) |
| **C** | Permutation-null для Cosine | Граница permutation: не работает для категориальных |

Приложения независимы. Читаются после основного нарратива — 
или пропускаются, если Time-детекторы не основная тема.

## Приложение A. SNR — вклад признаков


### Приложение A. SNR — вклад признаков

### Гипотеза

В `instrumentalness` визуально много шума. Возможно, SNR < 1.5 — 
признак перегружен случайными колебаниями.

### Метод

Signal-to-noise ratio для временного ряда:
- **Signal** = std rolling median (насколько меняется уровень 
  месяца к месяцу).
- **Noise** = median rolling std (насколько шумит внутри окна).
- Порог: SNR < 1.5 → признак шумный.

Плюс — permutation-тест: прирост SNR при сглаживании значим или 
в пределах шума?

### Зачем

Проверка, какие признаки дают сигнал, а какие — шум. Результат — 
диагностика, не фильтр. В фазе 4 SNR не используется как критерий — 
здесь показывается, почему.

In [17]:
# A. SNR — вклад признаков
# ============================================================================
# Signal-to-noise ratio для поведенческих признаков.
# Плюс permutation-тест: прирост SNR при сглаживании — сигнал или шум?
# ============================================================================
from diagnostics import compute_snr, snr_gain_permutation

BEH_FEATURES_SNR = ['hours', 'tracks', 'skip_rate', 'played_frac',
                    'desktop_share', 'obsession_share', 'discovery_rate']

# --- Собираем monthly_behavior для SNR ---
mb_snr = df_t.groupby('period').agg(
    hours=('ms_played', lambda x: x.sum() / 3_600_000),
    tracks=('spotify_track_uri', 'nunique'),
    skip_rate=('is_skip', 'mean'),
    played_frac=('played_frac', 'mean'),
    desktop_share=('platform', lambda x: x.fillna('').str.lower()
                                       .str.contains('windows|linux').mean()),
).sort_index()

# obsession_share (session-level — sess_flag построен в фазе 8)
mb_snr['obsession_share'] = (sess_flag.groupby('period')['is_obsession']
                              .mean()
                              .reindex(mb_snr.index).fillna(0.0))

# discovery_rate
first_seen_a = (df_t.groupby('spotify_track_uri')['ts_naive'].min()
                .dt.to_period('M'))
df_t['_first_month_a'] = df_t['spotify_track_uri'].map(first_seen_a)
df_t['_is_disc'] = df_t['period'] == df_t['_first_month_a']
mb_snr['discovery_rate'] = (df_t.groupby('period')['_is_disc']
                             .mean()
                             .reindex(mb_snr.index).fillna(0.0))

# --- 1. SNR до сглаживания ---
print("=" * 70)
print("SNR поведенческих признаков (до сглаживания)")
print("=" * 70)

snr_before = {}
for f in BEH_FEATURES_SNR:
    snr = compute_snr(mb_snr[f], window=3)
    snr_before[f] = snr
    flag = '⚠️' if snr < 1.5 else '✅'
    print(f"   {f:20s}: {snr:.3f} {flag}")

# --- 2. Permutation-тест ---
print("\n" + "=" * 70)
print("Permutation-тест SNR")
print("=" * 70)
print("Прирост SNR на реальных данных vs на случайных\n")

for f in ['tracks', 'skip_rate', 'played_frac',
          'discovery_rate', 'obsession_share']:
    real_gain, noise_p95, _ = snr_gain_permutation(
        mb_snr[f].dropna(), n_perm=200
    )
    verdict = ('✅ сигнал значим' if real_gain > noise_p95
               else '⚠️ в пределах шума')
    print(f"   {f:18s}: реальный {real_gain:+.3f}, "
          f"шум p95 {noise_p95:+.3f}  {verdict}")

# --- 3. SNR до / после сглаживания ---
print("\n" + "=" * 70)
print("SNR до и после сглаживания")
print("=" * 70)

print(f"\n{'Признак':<20} {'SNR до':<10} {'SNR после':<12} {'Δ'}")
print("-" * 55)

snr_after = {}
for f in BEH_FEATURES_SNR:
    if snr_before[f] < 1.5:
        smoothed = mb_snr[f].rolling(3, center=True, min_periods=2).mean()
        snr = compute_snr(smoothed, window=3)
        snr_after[f] = snr
        delta = snr - snr_before[f]
        mark = '✅' if snr >= 1.5 else '⚠️'
        print(f"{f:<20} {snr_before[f]:<10.3f} {snr:<12.3f} "
              f"{delta:+.3f} {mark}")
    else:
        snr_after[f] = snr_before[f]
        print(f"{f:<20} {snr_before[f]:<10.3f} {'—':<12} {'—'}")

SNR поведенческих признаков (до сглаживания)
   hours               : 1.641 ✅
   tracks              : 1.410 ⚠️
   skip_rate           : 1.375 ⚠️
   played_frac         : 1.075 ⚠️
   desktop_share       : 5.223 ✅
   obsession_share     : 1.300 ⚠️
   discovery_rate      : 1.129 ⚠️

Permutation-тест SNR
Прирост SNR на реальных данных vs на случайных

   tracks            : реальный +1.756, шум p95 +2.224  ⚠️ в пределах шума
   skip_rate         : реальный +1.088, шум p95 +1.352  ⚠️ в пределах шума
   played_frac       : реальный +1.026, шум p95 +1.360  ⚠️ в пределах шума
   discovery_rate    : реальный +1.262, шум p95 +1.255  ✅ сигнал значим
   obsession_share   : реальный +1.913, шум p95 +1.474  ✅ сигнал значим

SNR до и после сглаживания

Признак              SNR до     SNR после    Δ
-------------------------------------------------------
hours                1.641      —            —
tracks               1.410      3.166        +1.756 ✅
skip_rate            1.375      2.463        +1

### Результат

**SNR до сглаживания:**

| Признак | SNR до | Порог 1.5 |
|---|---|---|
| hours | 1.641 | ✅ |
| tracks | 1.410 | ⚠️ |
| skip_rate | 1.375 | ⚠️ |
| played_frac | 1.075 | ⚠️ |
| **desktop_share** | **5.223** | ✅ |
| obsession_share | 1.300 | ⚠️ |
| discovery_rate | 1.129 | ⚠️ |

**SNR после сглаживания (rolling 3 мес):**

| Признак | SNR до | SNR после | Δ |
|---|---|---|---|
| tracks | 1.410 | 3.166 | +1.756 |
| skip_rate | 1.375 | 2.463 | +1.088 |
| played_frac | 1.075 | 2.101 | +1.026 |
| obsession_share | 1.300 | 3.214 | +1.913 |
| discovery_rate | 1.129 | 2.392 | +1.262 |

Все сглаженные признаки перешли порог 1.5.

**Permutation-тест: прирост SNR не доказывает сигнал.**

| Признак | Реальный прирост | 95-й перцентиль шума | Вердикт |
|---|---|---|---|
| tracks | +1.756 | +2.224 | в пределах шума |
| skip_rate | +1.088 | +1.352 | в пределах шума |
| played_frac | +1.026 | +1.360 | в пределах шума |
| discovery_rate | +1.262 | +1.255 | на грани |
| obsession_share | +1.913 | +1.474 | **значим** |

Только `obsession_share` прошёл уверенно. `discovery_rate` — на грани 
(1.262 vs 1.255 (запас 0.007, на грани)). Остальные три — в пределах шума.

## 🔄 Поворот

**SNR — не критерий сигнала.**

Прирост SNR при сглаживании растёт независимо от того, есть ли сигнал. 
Для `tracks`, `skip_rate`, `played_frac` прирост на реальных данных 
не отличается от прироста на случайных. Это структурный дефект 
метрики — тот же тип, что self-percentile в фазе 6.

**Что в фазе 4 сделано правильно:** сглаживание оставлено — но по 
другому основанию. Не «SNR вырос», а «σ² в PELT-B упала вдвое 
(0.72 → 0.339), bootstrap-устойчивость выросла».

**Что это дало дальше:** в фазе 11 при формулировке toolkit SNR 
не используется — только permutation + практический гейт. 
Приложение показывает, почему.

**`desktop_share` — отдельный случай.** SNR = 5.223, но исключён 
не по SNR, а по природе: один структурный скачок Android → Windows, 
не разладка. В PELT такой признак давит остальные и смещает все 
точки к моменту смены платформы.

## Итог

- До сглаживания 5 из 7 признаков ниже порога 1.5; SNR не критерий, фильтрации по нему нет.
- SNR — плохая метрика для отбора. Заменена на σ² + bootstrap.
- Сглаживание оставлено — по конечному результату, не по SNR.

## Приложение B. Cut-scan + Storm — регионы нестабильности


### Приложение B. Cut-scan + Storm — регионы нестабильности

### Гипотеза

PELT ищет **точки** — конкретный месяц перелома. Но сигнал может 
быть **зонами**: переход растянут на несколько месяцев, PELT покажет 
одну точку в середине.

Два метода для поиска регионов:

**Cut-scan** — скользящая точка разреза. На каждом месяце считается 
разница медиан «до» и «после». Ищутся резкие изменения градиента — 
не самих дельт, а их скорости.

**Storm** — rolling std. Где профиль бежит — там std высокая. 
Эпизоды высокого std = зоны нестабильности.

### Зачем

1. **Дополнить PELT.** Точка внутри зоны — не «перелом», а «центр 
   перехода». Cut-scan покажет границы.
2. **Дать фоновые слои дашборду.** Точки PELT/cosine — вертикальные 
   линии. Зоны storm — заливка.
3. **Проверить гипотезу:** точки PELT/cosine попадают в зоны 
   нестабильности значимо чаще случайного?

### Метод проверки

Точки PELT/cosine — уникальные даты. Штормовые эпизоды покрывают 
часть истории. Считается число точек внутри штормовых месяцев. 
Сравнение с ожиданием при случайном покрытии. Биномиальный тест.

In [18]:
# B. Cut-scan + Storm
# ============================================================================
# Cut-scan: ищем регионы резкого изменения градиента Δ медиан.
# Storm: rolling std — зоны высокой нестабильности.
# Проверка: точки PELT/cosine значимо чаще внутри штормов?
# ============================================================================
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from scipy.stats import binomtest
from detectors.storm import detect as storm_detect

# --- 1. Cut-scan ---
SCAN_FEATURES = ['valence', 'energy', 'danceability', 'instrumentalness']
PRACTICAL_THRESHOLD = 0.02
MIN_WINDOW_MONTHS = 4
CLUSTER_GAP_MONTHS = 3

ms_scan = build_monthly_sound(df_t)
for f in SCAN_FEATURES:
    ms_scan[f] = ms_scan[f].fillna(ms_scan[f].median())

months_sorted = ms_scan.index.sort_values()
candidate_splits = months_sorted[MIN_WINDOW_MONTHS:-MIN_WINDOW_MONTHS]

rows = []
for split in candidate_splits:
    before = ms_scan[ms_scan.index < split]
    after = ms_scan[ms_scan.index >= split]
    if len(before) < MIN_WINDOW_MONTHS or len(after) < MIN_WINDOW_MONTHS:
        continue
    max_diff = max(
        abs(before[f].median() - after[f].median()) for f in SCAN_FEATURES
    )
    rows.append({'split': str(split), 'max_delta': max_diff})

scan = pd.DataFrame(rows).set_index('split')
scan['gradient'] = scan['max_delta'].diff().abs()

grad_valid = scan['gradient'].dropna()
med_grad = grad_valid.median()
mad_grad = np.median(np.abs(grad_valid - med_grad))
grad_threshold = med_grad + 1.5 * mad_grad

vals = scan['max_delta'].values
is_peak = np.zeros(len(vals), dtype=bool)
for i in range(1, len(vals) - 1):
    if vals[i] > vals[i - 1] and vals[i] > vals[i + 1]:
        is_peak[i] = True
scan['is_peak'] = is_peak

scan['is_break'] = (
    (scan['max_delta'] >= PRACTICAL_THRESHOLD) &
    ((scan['gradient'] > grad_threshold) | scan['is_peak'])
)
candidates = scan[scan['is_break']].copy()

# Кластеризация кандидатов
cand_months = sorted([pd.Period(s, 'M') for s in candidates.index])
clusters = []
if cand_months:
    current = [cand_months[0]]
    for m in cand_months[1:]:
        if (m - current[-1]).n <= CLUSTER_GAP_MONTHS:
            current.append(m)
        else:
            clusters.append(current)
            current = [m]
    clusters.append(current)

print("=" * 70)
print("CUT-SCAN: регионы")
print("=" * 70)
print(f"Порог градиента: {grad_threshold:.4f}")
print(f"Кандидатов: {len(candidates)}")
print(f"Регионов: {len(clusters)}")

cutscan_regions = []
for cl in clusters:
    cl_str = [str(m) for m in cl]
    sub = candidates.loc[[s for s in cl_str if s in candidates.index]]
    top = sub['max_delta'].idxmax()
    cutscan_regions.append({
        'peak': top,
        'region': f"{cl_str[0]}..{cl_str[-1]}" if len(cl_str) > 1 else cl_str[0],
        'n_candidates': len(cl_str),
        'max_delta': round(sub.loc[top, 'max_delta'], 3),
    })
    print(f"   {cutscan_regions[-1]['region']}  "
          f"(пик {top}, Δ={cutscan_regions[-1]['max_delta']}, "
          f"кандидатов={len(cl_str)})")

# --- 2. Storm ---
print("\n" + "=" * 70)
print("STORM: эпизоды")
print("=" * 70)

res_storm = storm_detect(ms_scan)

print(f"Порог: {res_storm['threshold']:.4f}")
print(f"Эпизодов: {len(res_storm['episodes'])}")
for ep in res_storm['episodes']:
    print(f"   {ep[0]} … {ep[-1]}  ({len(ep)} мес)")

# --- 3. Проверка: точки PELT/cosine в штормовых месяцах ---
print("\n" + "=" * 70)
print("ПРОВЕРКА: точки разладки в штормовых месяцах")
print("=" * 70)

pelt_a = ['2023-09', '2024-02', '2024-07']
pelt_b = ['2021-08', '2024-02', '2024-12', '2025-10', '2026-03']
cosine_points = ['2021-02', '2022-05', '2024-02']

all_points = sorted(set(pelt_a + pelt_b + cosine_points))
print(f"Всего уникальных точек: {len(all_points)}")
print(f"   {all_points}")

# ВАЖНО: штормовые месяцы — только те, что внутри episodes.
# episodes = список списков месяцев с is_storm=True (без пропусков).
storm_months = set()
for ep in res_storm['episodes']:
    for m in ep:
        storm_months.add(m)

coverage = len(storm_months) / len(ms_scan)
print(f"\nШтормовых месяцев: {len(storm_months)} из {len(ms_scan)} "
      f"({coverage:.1%})")

# Считаем попадания
points_inside = [p for p in all_points
                 if pd.Period(p, 'M') in storm_months]
n_inside = len(points_inside)
expected = coverage * len(all_points)

print(f"\nТочек внутри штормов: {n_inside} из {len(all_points)}")
print(f"   Точки в штормах: {points_inside}")
print(f"Ожидание при случайном: {expected:.2f}")

if n_inside > 0:
    result = binomtest(n_inside, len(all_points), coverage,
                       alternative='greater')
    print(f"\nБиномиальный тест:")
    print(f"   H0: точки попадают в штормы случайно (p = {coverage:.3f})")
    print(f"   Наблюдаем: {n_inside} из {len(all_points)}")
    print(f"   p-value = {result.pvalue:.4f}")
    if result.pvalue < 0.05:
        print(f"   ✅ Гипотеза подтверждена значимо")
    else:
        print(f"   ⚠️ В пределах случайного")

# Множество штормовых месяцев
storm_months = set()
for ep in res_storm['episodes']:
    for m in pd.period_range(ep[0], ep[-1], freq='M'):
        storm_months.add(m)

coverage = len(storm_months) / len(ms_scan)
print(f"Штормовых месяцев: {len(storm_months)} из {len(ms_scan)} "
      f"({coverage:.1%})")

# Считаем попадания
points_inside = [p for p in all_points
                 if pd.Period(p, 'M') in storm_months]
n_inside = len(points_inside)
expected = coverage * len(all_points)

print(f"\nТочек внутри штормов: {n_inside} из {len(all_points)}")
print(f"Ожидание при случайном: {expected:.1f}")

if n_inside > 0:
    result = binomtest(n_inside, len(all_points), coverage,
                       alternative='greater')
    print(f"\nБиномиальный тест:")
    print(f"   H0: точки попадают в штормы случайно (p = {coverage:.3f})")
    print(f"   p-value = {result.pvalue:.4f}")
    if result.pvalue < 0.05:
        print(f"   ✅ Гипотеза подтверждена значимо")
    else:
        print(f"   ⚠️ В пределах случайного")

# --- 4. Визуализация ---
fig = go.Figure()

# std_all — линия
x = ms_scan.index.to_timestamp()
# Пересчитаем std_all для визуализации
std_cols = []
ms_std = ms_scan.copy()
for f in SCAN_FEATURES:
    ms_std[f'{f}_std'] = ms_std[f].rolling(3, center=True,
                                            min_periods=2).std()
    std_cols.append(f'{f}_std')
ms_std['std_all'] = ms_std[std_cols].mean(axis=1)

fig.add_trace(go.Scatter(
    x=ms_std.index.to_timestamp(), y=ms_std['std_all'].values,
    mode='lines', name='std_all',
    line=dict(width=2),
))

fig.add_hline(
    y=res_storm['threshold'], line_dash='dash', line_color='gray',
    annotation_text=f'порог шторма {res_storm["threshold"]:.3f}',
    annotation_position='top right',
)

# Заливка штормов
for ep in res_storm['episodes']:
    fig.add_vrect(
        x0=pd.Period(ep[0], 'M').to_timestamp(),
        x1=pd.Period(ep[-1], 'M').to_timestamp(),
        fillcolor='rgba(255, 100, 100, 0.15)',
        line_width=0, layer='below',
    )

# Точки PELT/cosine
for p in pelt_a:
    fig.add_vline(x=pd.Period(p, 'M').to_timestamp(),
                  line_color='#1f77b4', line_width=1, line_dash='dot')
for p in pelt_b:
    fig.add_vline(x=pd.Period(p, 'M').to_timestamp(),
                  line_color='#ff7f0e', line_width=1, line_dash='dot')
for p in cosine_points:
    fig.add_vline(x=pd.Period(p, 'M').to_timestamp(),
                  line_color='#d62728', line_width=1, line_dash='dot')

fig.update_layout(
    template='plotly_white',
    height=400,
    title='Storm: эпизоды нестабильности + точки разладки',
    xaxis_title='Месяц',
    yaxis_title='std_all',
    margin=dict(l=60, r=40, t=60, b=40),
)
fig.show()

CUT-SCAN: регионы
Порог градиента: 0.0276
Кандидатов: 26
Регионов: 5
   2021-03..2021-06  (пик 2021-06, Δ=0.555, кандидатов=2)
   2021-12..2023-03  (пик 2022-09, Δ=0.482, кандидатов=13)
   2023-11..2024-07  (пик 2023-11, Δ=0.307, кандидатов=7)
   2025-02..2025-04  (пик 2025-02, Δ=0.164, кандидатов=2)
   2025-10..2026-01  (пик 2026-01, Δ=0.121, кандидатов=2)

STORM: эпизоды
Порог: 0.0996
Эпизодов: 2
   2021-01 … 2021-06  (3 мес)
   2023-10 … 2024-05  (5 мес)

ПРОВЕРКА: точки разладки в штормовых месяцах
Всего уникальных точек: 9
   ['2021-02', '2021-08', '2022-05', '2023-09', '2024-02', '2024-07', '2024-12', '2025-10', '2026-03']

Штормовых месяцев: 8 из 70 (11.4%)

Точек внутри штормов: 1 из 9
   Точки в штормах: ['2021-02']
Ожидание при случайном: 1.03

Биномиальный тест:
   H0: точки попадают в штормы случайно (p = 0.114)
   Наблюдаем: 1 из 9
   p-value = 0.6645
   ⚠️ В пределах случайного
Штормовых месяцев: 14 из 70 (20.0%)

Точек внутри штормов: 2 из 9
Ожидание при случайном: 1.8



### Результат

**Cut-scan: 5 регионов** — данные меняются не за месяц, а за 3–15.

| Регион | Пик | Δ | Кандидатов |
|---|---|---|---|
| 2021-03 .. 2021-06 | 2021-06 | 0.555 | 2 |
| 2021-12 .. 2023-03 | 2022-09 | 0.482 | 13 |
| 2023-11 .. 2024-07 | 2023-11 | 0.307 | 7 |
| 2025-02 .. 2025-04 | 2025-02 | 0.164 | 2 |
| 2025-10 .. 2026-01 | 2026-01 | 0.121 | 2 |

**Storm: 2 эпизода** высокого `std_all`.

| Эпизод | Штормовых месяцев |
|---|---|
| 2021-01, 2021-02, 2021-06 | 3 |
| 2023-10, 2024-01, 2024-03, 2024-04, 2024-05 | 5 |

Итого **8 штормовых месяцев из 70 (11.4%)**. Порог `std_all > 0.0996`.

**Проверка гипотезы: точки PELT/cosine в штормах?**

| Метрика | Значение |
|---|---|
| Уникальных точек | 9 |
| Штормовых месяцев | 8 из 70 (11.4%) |
| Точек в штормах | **1 из 9** |
| Ожидание при случайном | 1.03 |
| Биномиальный p-value | **0.6645** |

Точка в шторме — только `2021-02` (Cosine).

**Второй прогон: заливка по диапазону эпизода** (как в дашборде).

| Метрика | Значение |
|---|---|
| Штормовых месяцев | 14 из 70 (20.0%) |
| Точек внутри | 2 из 9 |
| Ожидание при случайном | 1.8 |
| Биномиальный p-value | 0.5638 |

Вывод тот же: в пределах случайного.

### 🔄 Поворот

**1. Гипотеза «шторм объясняет точки» — не подтвердилась.**

`1 из 9` при ожидании `1.03`. `p = 0.66` — в пределах случайного. 
Точки PELT/cosine **не концентрируются** в штормовых зонах.

Это не баг. Это **свойство данных**: слои отвечают на разные вопросы. 
Точки — «где переломилось». Шторм — «где профиль бежал быстрее». 
Пересечение не обязательно.

**2. Cut-scan и Storm — разные масштабы.**

- **Cut-scan** даёт длинные регионы: 3–15 месяцев. Самый крупный — 
  `2021-12..2023-03` с 13 кандидатами внутри.
- **Storm** даёт плотные эпизоды: 3 и 5 штормовых месяцев.

Пересечение: `2023-11..2024-07` (cut-scan) ∩ `2023-10..2024-05` 
(storm). Один эпизод, разные методы видят его на разной длине.

**3. Заливка эпизодов — по диапазону, не по штормовым месяцам.**

Storm-эпизод в дашборде закрашивается от первого до последнего 
месяца эпизода. Внутри могут быть **пропуски** — месяцы, где 
`is_storm=False`.

Пример: эпизод `2023-10..2024-05` закрашивается целиком, но 
штормовых месяцев внутри — только 5 из 8. Точка `2024-02` (Cosine) 
визуально внутри заливки, формально — на пропуске.

### Зачем это приложение

**Обосновать фоновый слой Storm.** Заливка штормов в дашборде получена этим методом.

**Плюс** — проверка гипотезы «шторм = детонатор для точки». 
Проверена, отвергнута.

### Итог

- **Cut-scan** — исследовательский слой регионов; в дашборд не идёт.
- **Storm** — идёт в продакшн как `time_storms.parquet` (линия `std_all` и заливка эпизодов).
- Слои независимы от точек PELT/cosine: «что переломилось» и «где бежало» — разные измерения.
- Гипотеза «точки в штормах значимо чаще» отвергнута (1 из 9, p = 0.66; по диапазону 2 из 9, p = 0.56).

## Приложение C. Границы permutation



### Приложение C. Cosine-null — граница permutation

### Гипотеза

В фазе 11 сформулирован ключевой принцип: **«единица обмена»** — 
что можно перемешивать без потери смысла. Toolkit делает это 
явным параметром.

Приложение показывает **случай, где принцип нарушается** — 
permutation для Cosine ломается.

**Зачем:** показать **границу применимости** permutation. Метод 
работает для непрерывных величин с event-структурой. Не для всего.

### Метод

Permutation-подход из Variability применяется к косинусному 
расстоянию на артистах. Для каждой пары соседних месяцев:
1. Считается наблюдаемое расстояние.
2. Null: перемешивание событий двух месяцев → пересчёт shares → 
   расстояние.
3. Сравнение: `observed > p95 null`?

Проверка: сколько переходов окажутся значимыми.

In [19]:
# C. Permutation-null для Cosine (быстрая версия)
# ============================================================================
# Векторизованный подсчёт: вместо (a['artist_name'] == art).sum()
# используем pd.value_counts() один раз на итерацию.
# n_perm снижен до 50 — для оценки хватает.
# ============================================================================
import numpy as np
import pandas as pd

def monthly_share_matrix_c(df_in, id_col, min_events=10):
    d = df_in[['period', id_col]].dropna().copy()
    d = d[d[id_col] != 'unknown']
    sizes = d.groupby('period').size()
    valid = sizes[sizes >= min_events].index
    d = d[d['period'].isin(valid)]
    counts = d.groupby(['period', id_col]).size().unstack(fill_value=0)
    return counts.div(counts.sum(axis=1), axis=0).sort_index()


def cosine_dist_c(a, b):
    denom = np.linalg.norm(a) * np.linalg.norm(b)
    return float(1 - np.dot(a, b) / denom) if denom > 0 else np.nan


print("=" * 70)
print("C3. PERMUTATION-NULL ДЛЯ COSINE")
print("=" * 70)

shares_c = monthly_share_matrix_c(df_t, 'artist_name')
months_c = list(shares_c.index)
rng_c = np.random.default_rng(42)

events_artist = df_t[['period', 'artist_name']].dropna().copy()
events_artist = events_artist[events_artist['artist_name'] != 'unknown']

n_perm = 50
results = []

for i in range(1, len(months_c)):
    m0, m1 = months_c[i - 1], months_c[i]

    obs = cosine_dist_c(shares_c.loc[m0].values, shares_c.loc[m1].values)

    e0 = events_artist[events_artist['period'] == m0]
    e1 = events_artist[events_artist['period'] == m1]
    n0, n1 = len(e0), len(e1)
    if n0 < 5 or n1 < 5:
        continue

    # Объединённый массив artist_name (быстрее, чем pandas)
    pooled_arr = pd.concat([e0['artist_name'], e1['artist_name']],
                            ignore_index=True).values
    pooled_codes = pd.Categorical(pooled_arr).codes
    n_cats = pooled_codes.max() + 1

    nulls = np.empty(n_perm)
    for k in range(n_perm):
        perm = rng_c.permutation(len(pooled_codes))
        a_codes = pooled_codes[perm[:n0]]
        b_codes = pooled_codes[perm[n0:n0 + n1]]

        # Быстрый подсчёт через np.bincount
        va = np.bincount(a_codes, minlength=n_cats).astype(float)
        vb = np.bincount(b_codes, minlength=n_cats).astype(float)
        va /= va.sum() or 1
        vb /= vb.sum() or 1

        nulls[k] = cosine_dist_c(va, vb)

    thr = np.percentile(nulls, 95)
    results.append({
        'month': str(m1),
        'observed': obs,
        'null_p95': thr,
        'significant': obs > thr,
    })

res_df = pd.DataFrame(results)

print(f"\nВсего переходов: {len(res_df)}")
print(f"Значимых: {res_df['significant'].sum()} "
      f"({res_df['significant'].mean():.1%})")
print(f"Ожидание при случайном: 5%")

print(f"\nПервые 10 переходов:")
print(res_df.head(10).round(3).to_string(index=False))

C3. PERMUTATION-NULL ДЛЯ COSINE

Всего переходов: 69
Значимых: 69 (100.0%)
Ожидание при случайном: 5%

Первые 10 переходов:
  month  observed  null_p95  significant
2020-11     0.562     0.219         True
2020-12     0.708     0.513         True
2021-01     0.685     0.513         True
2021-02     0.993     0.035         True
2021-03     0.958     0.159         True
2021-04     0.642     0.157         True
2021-05     0.856     0.164         True
2021-06     0.890     0.169         True
2021-07     0.688     0.139         True
2021-08     0.623     0.134         True


### Результат

**Все 69 переходов значимы** при null p95. Ожидание при случайном — 5%.

| Метрика | Значение |
|---|---|
| Всего переходов | 69 |
| Значимых | **69 (100%)** |
| Ожидание | 5% |

**Разрыв `observed` vs `null_p95` — огромный.**

| Месяц | observed | null p95 | Ratio |
|---|---|---|---|
| 2021-02 | 0.993 | 0.035 | 28× |
| 2021-03 | 0.958 | 0.159 | 6× |
| 2020-12 | 0.708 | 0.513 | 1.4× |
| 2020-11 | 0.562 | 0.219 | 2.6× |

Ни один переход не близок к границе.

### 🔄 Поворот

**Permutation-null не работает для Cosine.**

**Все 69 переходов значимы.** Не «большинство» — **все**. Это 
признак структурного провала: тест не разделяет сигнал и шум.

**Причина — свойство данных, не метода.**

Перемешивание событий двух соседних месяцев даёт **почти 
одинаковые** распределения. Закон больших чисел: 1500 + 1500 
случайных событий → оба подмножества покрывают ~одинаковый набор 
артистов. Null distance = 0.1–0.2.

Реальные соседние месяцы — **всегда разные**. `observed` = 0.5–0.9.

**Нулевая гипотеза неправдоподобна.** «Состав артистов не изменился» 
— для этой структуры данных **никогда не верно**. Permutation 
проверяет гипотезу, которую данные отвергают априори.

**Это не проблема permutation как метода.** Это **граница 
применимости**. Permutation работает там, где можно построить 
**правдоподобную** нулевую гипотезу:

| Данные | Нулевая гипотеза | Работает? |
|---|---|---|
| Sound (valence, energy) | «уровень не изменился» | ✅ правдоподобна |
| Behavior (skip_rate) | «доля не изменилась» | ✅ правдоподобна |
| **Cosine (состав артистов)** | **«состав не изменился»** | ❌ неправдоподобна |

Для непрерывных величин «нет изменения» — осмысленная гипотеза. 
Для категориальных с высокой кардинальностью — **нет**.

**Что используем вместо этого.** Для Cosine — **MAD-порог**: 
`медиана + 2·MAD`. Это **outlier detection**, а не проверка 
гипотезы. Ищем **выбросы** на фоне обычной вариативности, а не 
«значимое отличие от нуля».

MAD-порог работает: даёт 3 устойчивых пика (`2021-02`, `2022-05`, 
`2024-02`). Не «всё значимо», а «вот конкретные выделяющиеся точки».

### Зачем это приложение

**Показать границу применимости permutation.** В фазе 11 мы 
сформулировали «единицу обмена» как ключевой принцип. Здесь — 
**эмпирическая проверка** этого принципа:

- Где единица обмена осмысленна → permutation работает.
- Где нет → **все переходы значимы**, метод бесполезен.

**Cosine — единственный случай в проекте, где permutation ломается.** 
Не потому что метод плохой — потому что **нулевая гипотеза 
неправдоподобна для этой структуры данных**.

### Итог

- **Permutation-null для Cosine отвергнут.** 69 из 69 значимы.
- **MAD-порог идёт в продакшн** для Cosine — outlier detection, 
  не проверка гипотезы.
- **Граница permutation** зафиксирована: работает для непрерывных 
  величин с event-структурой. Не для категориальных.
- **Приложение завершает тему** «единицы обмена» из фазы 11 — 
  показывает, где принцип нарушается.

#

#